# 2023~2025년 CASE 1·2 Base/Full 분류모형

## 분석 목적과 해석 범위

국민여행조사에서 다음 두 대표 여행 유형을 구분하는 연관 특성을
Base와 Full 피처 계약으로 비교한다.

- CASE 1(타깃 0): 국내 관광·휴양 여행
- CASE 2(타깃 1): 관광·휴양 활동을 포함한 가족·친지·친구 방문
- 분석 단위: 응답자·조사회차 1행
- 키: `YEAR + ID`
- `WT_DOM`: 품질 확인용으로만 보존하며 학습·분할·평가에는 미사용

Base는 의미 중심 요약 피처, Full은 Base에 세부 이동·지역·숙박·예약·
활동·정보·동반·지출 피처를 추가한다. Full은 Base에서 고른 같은
알고리즘의 하이퍼파라미터를 재사용하므로 별도 탐색하지 않는다.

> 이 노트북은 CASE 1·2와 **연관된 특성**을 설명한다. 예측 성능과 SHAP을
> 인과효과로 해석하지 않는다. 가족·친지·친구 방문은 출생지나 법적
> 고향을 직접 측정한 개념이 아니다.

## 1. 실행 환경과 재현성 설정

최상단 설정만 바꾸어 로컬 `tourism` 환경과 Google Colab에서 같은
분석을 실행한다. `RESUME_RUN_DIR`에는 이전 실행의 출력 루트를 지정한다.
Colab에서는 Google Drive의 `data`, `notebooks`, `outputs` 구조를 쓴다.

In [3]:
from pathlib import Path


COLAB_MODE = True
COLAB_DATA_DIR = Path("/content/drive/MyDrive/tourism_poster/data")
RESUME_RUN_DIR = None
RANDOM_STATE = 20260726
SEARCH_CANDIDATES = 20
CV_FOLDS = 3
SHAP_SAMPLE_SIZE = 2_000
BOOTSTRAP_ITERATIONS = 2_000

NOTEBOOK_STEM = "260726_CASE1_CASE2_분류모형"
NOTEBOOK_RELATIVE_PATH = (
    "notebooks/03_Classification/260726_CASE1_CASE2_분류모형.ipynb"
)
EXPECTED_ANALYSIS_N = 70_694
EXPECTED_CASE1_N = 57_904
EXPECTED_CASE2_N = 12_790

print("COLAB_MODE:", COLAB_MODE)

COLAB_MODE: True


In [4]:
import importlib.util
import subprocess
import sys


if COLAB_MODE:
    from google.colab import drive

    drive.mount("/content/drive")
    required_packages = {
        "xgboost": "xgboost",
        "lightgbm": "lightgbm",
        "catboost": "catboost",
        "shap": "shap",
        "pyarrow": "pyarrow",
        "holidays": "holidays",
    }
    missing_packages = [
        package
        for module, package in required_packages.items()
        if importlib.util.find_spec(module) is None
    ]
    if missing_packages:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", *missing_packages]
        )
    PROJECT_ROOT = COLAB_DATA_DIR.parent
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "src").is_dir():
        if PROJECT_ROOT.parent == PROJECT_ROOT:
            raise FileNotFoundError(
                "src 폴더가 있는 프로젝트 루트를 찾지 못했습니다."
            )
        PROJECT_ROOT = PROJECT_ROOT.parent
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

Mounted at /content/drive


In [6]:
import json
import os
import platform
import re
import tempfile
import warnings
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version

import holidays
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from catboost import CatBoostClassifier
from IPython.display import display
from lightgbm import LGBMClassifier
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    ParameterSampler,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*does not have valid feature names.*")

if COLAB_MODE:
    DATA_DIR = COLAB_DATA_DIR
    PREPROCESS_DATA_DIR = DATA_DIR / "preprocess"
    RAW_DATA_DIR = DATA_DIR / "raw"
    OUTPUTS_DIR = DATA_DIR.parent / "outputs"
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR = (
        PREPROCESS_DATA_DIR / "national_travel_survey"
    )
    NATIONAL_TRAVEL_SURVEY_RAW_DATA_DIR = (
        RAW_DATA_DIR / "national_travel_survey"
    )
    COLORS = {
        "primary": "#2F5D8C",
        "secondary": "#2A9D8F",
        "accent": "#E76F51",
        "highlight": "#E9C46A",
        "neutral": "#6B7280",
        "negative": "#C44E52",
        "background": "#FAFAF8",
        "ink": "#252A31",
        "grid": "#D9DDE3",
    }
    PALETTE = tuple(
        COLORS[name]
        for name in [
            "primary",
            "secondary",
            "accent",
            "highlight",
            "neutral",
        ]
    )


    def apply_plot_style() -> None:
        '''Colab에서 프로젝트와 같은 시각화 스타일을 적용한다.'''
        font_files = sorted(
            (
                DATA_DIR.parent
                / "font"
                / "Pretendard-1.3.9"
                / "public"
                / "static"
            ).glob("Pretendard-*.otf")
        )
        if font_files:
            from matplotlib import font_manager

            for font_path in font_files:
                font_manager.fontManager.addfont(str(font_path))
            font_family = font_manager.FontProperties(
                fname=str(font_files[0])
            ).get_name()
        else:
            font_family = "DejaVu Sans"
        plt.rcParams.update(
            {
                "font.family": font_family,
                "font.size": 11,
                "axes.titlesize": 13,
                "axes.spines.top": False,
                "axes.spines.right": False,
                "axes.unicode_minus": False,
                "figure.facecolor": COLORS["background"],
                "axes.facecolor": COLORS["background"],
                "savefig.facecolor": COLORS["background"],
            }
        )
else:
    from src.path import (
        NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR,
        NATIONAL_TRAVEL_SURVEY_RAW_DATA_DIR,
        PREPROCESS_DATA_DIR,
        create_output_directories,
    )
    from src.visualization import COLORS, PALETTE, apply_plot_style

apply_plot_style()

In [7]:
if RESUME_RUN_DIR is not None:
    RUN_DIR = Path(RESUME_RUN_DIR).expanduser().resolve()
    FIGURES_DIR = RUN_DIR / "figures"
    TABLES_DIR = RUN_DIR / "tables"
elif COLAB_MODE:
    run_timestamp = datetime.now().strftime("%y%m%d_%H%M")
    RUN_DIR = OUTPUTS_DIR / f"{NOTEBOOK_STEM}_{run_timestamp}"
    FIGURES_DIR = RUN_DIR / "figures"
    TABLES_DIR = RUN_DIR / "tables"
else:
    FIGURES_DIR, TABLES_DIR = create_output_directories(
        f"{NOTEBOOK_STEM}.ipynb"
    )
    RUN_DIR = FIGURES_DIR.parent

MODELS_DIR = RUN_DIR / "models"
for directory in (FIGURES_DIR, TABLES_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DATA_PATH = (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR
    / "national_travel_survey_2023_2025_preprocessed.csv"
)
CODEBOOK_PATH = (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR
    / "national_travel_survey_2023_2025_preprocessed_codebook.csv"
)
REGION_CODE_PATH = (
    NATIONAL_TRAVEL_SURVEY_RAW_DATA_DIR
    / "national_travel_survey_region_code.csv"
)
CLASSIFICATION_DIR = PREPROCESS_DATA_DIR / "classification"
BASE_PARQUET_PATH = CLASSIFICATION_DIR / "case1_case2_base.parquet"
FULL_PARQUET_PATH = CLASSIFICATION_DIR / "case1_case2_full.parquet"
FEATURE_DICTIONARY_PATH = CLASSIFICATION_DIR / "feature_dictionary.csv"
EXCLUSION_LOG_PATH = CLASSIFICATION_DIR / "exclusion_log.csv"
SPLIT_ASSIGNMENT_PATH = CLASSIFICATION_DIR / "split_assignment.csv"
PREPROCESSING_ARTIFACT_PATHS = [
    BASE_PARQUET_PATH,
    FULL_PARQUET_PATH,
    FEATURE_DICTIONARY_PATH,
    EXCLUSION_LOG_PATH,
    SPLIT_ASSIGNMENT_PATH,
]

print(f"실행 모드: {'Colab' if COLAB_MODE else '로컬'}")
print(f"입력 데이터: {DATA_PATH}")
print(f"실행 출력 루트: {RUN_DIR}")

실행 모드: Colab
입력 데이터: /content/drive/MyDrive/tourism_poster/data/preprocess/national_travel_survey/national_travel_survey_2023_2025_preprocessed.csv
실행 출력 루트: /content/drive/MyDrive/tourism_poster/outputs/260726_CASE1_CASE2_분류모형_260725_2038


In [8]:
PACKAGE_NAMES = [
    "numpy",
    "pandas",
    "scikit-learn",
    "xgboost",
    "lightgbm",
    "catboost",
    "shap",
    "pyarrow",
    "holidays",
    "joblib",
]


def package_versions(package_names: list[str]) -> dict[str, str]:
    '''설치된 패키지 버전을 조회한다.

    Args:
        package_names: 배포 패키지 이름 목록.

    Returns:
        패키지별 설치 버전. 조회되지 않으면 ``not-installed``.
    '''
    versions = {}
    for package_name in package_names:
        try:
            versions[package_name] = version(package_name)
        except PackageNotFoundError:
            versions[package_name] = "not-installed"
    return versions


run_metadata = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "colab_mode": COLAB_MODE,
    "random_state": RANDOM_STATE,
    "search_candidates": SEARCH_CANDIDATES,
    "cv_folds": CV_FOLDS,
    "shap_sample_size": SHAP_SAMPLE_SIZE,
    "packages": package_versions(PACKAGE_NAMES),
}

## 2. 공통 저장 함수

CSV·JSON·Parquet·joblib·NPZ는 같은 디렉터리의 임시 파일을 완성한 뒤
교체한다. 장시간 탐색 중에도 완료된 fold와 모델이 손상된 반쪽 파일로
남지 않게 한다.

In [9]:
def _temporary_path(path: Path) -> Path:
    '''원자적 교체에 사용할 임시 경로를 만든다.

    Args:
        path: 최종 파일 경로.

    Returns:
        최종 파일과 같은 디렉터리의 임시 경로.
    '''
    path.parent.mkdir(parents=True, exist_ok=True)
    file_descriptor, temporary_name = tempfile.mkstemp(
        dir=path.parent,
        prefix=f".{path.name}.",
        suffix=".tmp",
    )
    os.close(file_descriptor)
    return Path(temporary_name)


def atomic_csv(
    frame: pd.DataFrame,
    path: Path,
    encoding: str = "utf-8-sig",
) -> None:
    '''DataFrame을 CSV로 원자적으로 저장한다.

    Args:
        frame: 저장할 DataFrame.
        path: 최종 CSV 경로.
        encoding: 출력 인코딩.
    '''
    temporary_path = _temporary_path(path)
    try:
        frame.to_csv(temporary_path, index=False, encoding=encoding)
        os.replace(temporary_path, path)
    finally:
        temporary_path.unlink(missing_ok=True)


def atomic_json(payload: dict, path: Path) -> None:
    '''사전을 JSON으로 원자적으로 저장한다.

    Args:
        payload: 저장할 JSON 객체.
        path: 최종 JSON 경로.
    '''
    temporary_path = _temporary_path(path)
    try:
        temporary_path.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2, default=str),
            encoding="utf-8",
        )
        os.replace(temporary_path, path)
    finally:
        temporary_path.unlink(missing_ok=True)


def atomic_parquet(frame: pd.DataFrame, path: Path) -> None:
    '''DataFrame을 Parquet으로 원자적으로 저장한다.

    Args:
        frame: 저장할 DataFrame.
        path: 최종 Parquet 경로.
    '''
    temporary_path = _temporary_path(path)
    try:
        frame.to_parquet(temporary_path, index=False)
        os.replace(temporary_path, path)
    finally:
        temporary_path.unlink(missing_ok=True)


def atomic_joblib(value: object, path: Path) -> None:
    '''Python 객체를 joblib 파일로 원자적으로 저장한다.

    Args:
        value: 직렬화할 객체.
        path: 최종 joblib 경로.
    '''
    temporary_path = _temporary_path(path)
    try:
        joblib.dump(value, temporary_path)
        os.replace(temporary_path, path)
    finally:
        temporary_path.unlink(missing_ok=True)


def atomic_npz(path: Path, **arrays: np.ndarray) -> None:
    '''배열 묶음을 압축 NPZ로 원자적으로 저장한다.

    Args:
        path: 최종 NPZ 경로.
        **arrays: 이름별 저장 배열.
    '''
    temporary_path = _temporary_path(path)
    try:
        with temporary_path.open("wb") as file:
            np.savez_compressed(file, **arrays)
        os.replace(temporary_path, path)
    finally:
        temporary_path.unlink(missing_ok=True)


atomic_json(run_metadata, MODELS_DIR / "run_metadata.json")

## 3. 데이터 로드와 분석 단위 확인

통합 CSV와 코드북의 열을 1:1로 대조하고, `YEAR + ID` 유일성,
`WT_DOM`의 결측·양수 여부, CASE 1·2 표본 수를 확인한다. 통합본은
응답자·조사회차 한 행이며, 방문지 슬롯은 행 내부의 반복 열이다.

In [10]:
raw_data = pd.read_csv(DATA_PATH, low_memory=False)
codebook = pd.read_csv(CODEBOOK_PATH)
region_codes = pd.read_csv(REGION_CODE_PATH, encoding="utf-8")

expected_columns = codebook["column_name"].tolist()
assert len(expected_columns) == len(set(expected_columns)), (
    "코드북 column_name이 중복되었습니다."
)
assert set(raw_data.columns) == set(expected_columns), (
    "통합 데이터 열과 코드북 행이 1:1로 대응하지 않습니다."
)
assert not raw_data.duplicated(["YEAR", "ID"]).any(), (
    "YEAR + ID 키가 중복되었습니다."
)
assert raw_data["WT_DOM"].notna().all(), "WT_DOM에 결측이 있습니다."
assert raw_data["WT_DOM"].gt(0).all(), "WT_DOM에 0 이하 값이 있습니다."

raw_data["D_TRA_CASE"] = pd.to_numeric(
    raw_data["D_TRA_CASE"],
    errors="coerce",
)
analysis_data = raw_data.loc[
    raw_data["D_TRA_CASE"].isin([1, 2])
].copy()
analysis_data["target"] = (
    analysis_data["D_TRA_CASE"].eq(2).astype("int8")
)

target_counts = analysis_data["target"].value_counts().sort_index()
assert len(analysis_data) == EXPECTED_ANALYSIS_N
assert target_counts.to_dict() == {
    0: EXPECTED_CASE1_N,
    1: EXPECTED_CASE2_N,
}
print(
    f"분석 표본: {len(analysis_data):,}행, "
    f"CASE 2 비율: {analysis_data['target'].mean():.1%}"
)

분석 표본: 70,694행, CASE 2 비율: 18.1%


In [11]:
sample_flow = pd.DataFrame(
    {
        "단계": [
            "전체 응답자·조사회차",
            "CASE 1·2 분석 표본",
            "CASE 1",
            "CASE 2",
        ],
        "비가중_N": [
            len(raw_data),
            len(analysis_data),
            analysis_data["target"].eq(0).sum(),
            analysis_data["target"].eq(1).sum(),
        ],
    }
)
year_case = (
    analysis_data.groupby(["YEAR", "target"], observed=True)
    .size()
    .rename("비가중_N")
    .reset_index()
)
year_case["CASE"] = year_case["target"].map(
    {0: "CASE 1", 1: "CASE 2"}
)
year_case["연도내_비율"] = year_case["비가중_N"] / year_case.groupby(
    "YEAR"
)["비가중_N"].transform("sum")

display(sample_flow)
display(year_case[["YEAR", "CASE", "비가중_N", "연도내_비율"]])

,단계,비가중_N
0,전체 응답자·조사회차,156050
1,CASE 1·2 분석 표본,70694
2,CASE 1,57904
3,CASE 2,12790


,YEAR,CASE,비가중_N,연도내_비율
0,2023,CASE 1,19446,0.814390
1,2023,CASE 2,4432,0.185610
2,2024,CASE 1,18917,0.806489
3,2024,CASE 2,4539,0.193511
4,2025,CASE 1,19541,0.836515
5,2025,CASE 2,3819,0.163485


## 4. 코드북 기반 구조적 비해당 복원

`structural_missing_code`에 기록된 코드를 일반 결측과 구분해 복원한다.
CASE 1·2 필터 후 전부 결측인 원열은 후보에서 제외하되, 어떤 열이
제외됐는지 로그에 남긴다.

In [13]:
structural_rows = codebook.loc[
    codebook["structural_missing_code"].notna(),
    ["column_name", "structural_missing_code"],
]
structural_code_pattern = re.compile(r"^\s*(-?\d+(?:\.\d+)?)\s*=")

structural_replacements = []
unparsed_structural_codes = []

for row in structural_rows.itertuples(index=False):
    if row.column_name not in analysis_data.columns:
        continue

    structural_code_text = str(row.structural_missing_code)
    match = structural_code_pattern.match(structural_code_text)

    if match is None:
        unparsed_structural_codes.append(
            {
                "column_name": row.column_name,
                "structural_missing_code": structural_code_text,
            }
        )
        continue

    missing_code = float(match.group(1))
    values = pd.to_numeric(
        analysis_data[row.column_name],
        errors="coerce",
    )
    replacement_count = int(values.eq(missing_code).sum())
    analysis_data[row.column_name] = values.mask(values.eq(missing_code))

    structural_replacements.append(
        {
            "column_name": row.column_name,
            "structural_missing_code": missing_code,
            "replaced_N": replacement_count,
        }
    )

empty_after_filter = analysis_data.columns[
    analysis_data.isna().all()
].tolist()

print(
    f"구조적 비해당 처리 열: {len(structural_replacements):,}개, "
    f"코드 해석 불가 열: {len(unparsed_structural_codes):,}개, "
    f"CASE 1·2 전부 결측 열: {len(empty_after_filter):,}개"
)

if unparsed_structural_codes:
    display(pd.DataFrame(unparsed_structural_codes))

구조적 비해당 처리 열: 0개, 코드 해석 불가 열: 15개, CASE 1·2 전부 결측 열: 51개


,column_name,structural_missing_code
0,D_TRA_B7_1,#NAME?
1,D_TRA_B7_2,#NAME?
2,D_TRA_B7_3,#NAME?
3,D_TRA_B7_RANKW_1,#NAME?
4,D_TRA_B7_RANKW_2,#NAME?
5,D_TRA_B7_RANKW_3,#NAME?
6,D_TRA_B7_RANKW_4,#NAME?
7,D_TRA_B7_RANKW_5,#NAME?
8,D_TRA_B7_RANKW_6,#NAME?
9,D_TRA_B7_RANKW_7,#NAME?


## 5. Base 행 단위 파생

행 단위 의미 요약은 분할 전에 만든다. 날짜·일수·동반·이동·방문지역·
숙박·예약·정보·활동·지출을 타깃과 무관하게 변환한다. 지출의 99%
상한과 로그 변환은 여기서 확정하지 않고 이후 Pipeline이 Train에서만
학습한다.

In [14]:
def numeric_frame(
    frame: pd.DataFrame,
    columns: list[str],
) -> pd.DataFrame:
    '''존재하는 열을 수치형 DataFrame으로 변환한다.

    Args:
        frame: 원본 DataFrame.
        columns: 변환할 후보 열.

    Returns:
        존재하는 후보 열로 구성한 수치형 DataFrame.
    '''
    existing = [column for column in columns if column in frame.columns]
    return frame[existing].apply(pd.to_numeric, errors="coerce")


def any_code(frame: pd.DataFrame, codes: set[int]) -> pd.Series:
    '''행마다 지정 코드가 하나라도 있는지 판정한다.

    Args:
        frame: 수치형 코드 DataFrame.
        codes: 선택으로 볼 코드 집합.

    Returns:
        행별 0/1 Series.
    '''
    return frame.isin(codes).any(axis=1).astype("int8")


def any_selected(frame: pd.DataFrame) -> pd.Series:
    '''0/1 항목 중 하나라도 선택됐는지 판정한다.

    Args:
        frame: 항목별 0/1 DataFrame.

    Returns:
        행별 0/1 Series.
    '''
    return frame.eq(1).any(axis=1).astype("int8")


analysis_data["start_date"] = pd.to_datetime(
    {
        "year": pd.to_numeric(
            analysis_data["D_TRA_SYEAR"],
            errors="coerce",
        ),
        "month": pd.to_numeric(
            analysis_data["D_TRA_SMONTH"],
            errors="coerce",
        ),
        "day": pd.to_numeric(
            analysis_data["D_TRA_SDAY"],
            errors="coerce",
        ),
    },
    errors="coerce",
)
analysis_data["end_date"] = pd.to_datetime(
    {
        "year": pd.to_numeric(
            analysis_data["D_TRA_EYEAR"],
            errors="coerce",
        ),
        "month": pd.to_numeric(
            analysis_data["D_TRA_EMONTH"],
            errors="coerce",
        ),
        "day": pd.to_numeric(
            analysis_data["D_TRA_EDAY"],
            errors="coerce",
        ),
    },
    errors="coerce",
)
analysis_data["season"] = analysis_data["start_date"].dt.month.map(
    {
        12: "겨울",
        1: "겨울",
        2: "겨울",
        3: "봄",
        4: "봄",
        5: "봄",
        6: "여름",
        7: "여름",
        8: "여름",
        9: "가을",
        10: "가을",
        11: "가을",
    }
)
analysis_data["nights"] = pd.to_numeric(
    analysis_data["D_TRA_S_Day"],
    errors="coerce",
).clip(lower=0)
analysis_data["trip_days"] = analysis_data["nights"] + 1
analysis_data["overnight_type"] = np.where(
    analysis_data["nights"].gt(0),
    "숙박",
    "당일",
)
analysis_data["trip_days_band"] = pd.cut(
    analysis_data["trip_days"],
    bins=[0, 1, 2, 3, np.inf],
    labels=["1일", "2일", "3일", "4일 이상"],
).astype("object")

In [15]:
korean_holidays = holidays.KR(years=[2023, 2024, 2025, 2026])


def calendar_flags(
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> tuple[float, float, float]:
    '''여행 기간의 평일·주말·공휴일 포함 여부를 계산한다.

    Args:
        start: 여행 시작일.
        end: 여행 종료일.

    Returns:
        평일, 주말, 공휴일 포함 여부.
    '''
    if pd.isna(start) or pd.isna(end) or end < start:
        return np.nan, np.nan, np.nan
    dates = pd.date_range(
        start,
        min(end, start + pd.Timedelta(days=60)),
    )
    return (
        float(any(date.weekday() < 5 for date in dates)),
        float(any(date.weekday() >= 5 for date in dates)),
        float(any(date.date() in korean_holidays for date in dates)),
    )


calendar_values = [
    calendar_flags(start, end)
    for start, end in zip(
        analysis_data["start_date"],
        analysis_data["end_date"],
    )
]
analysis_data[
    ["includes_weekday", "includes_weekend", "includes_holiday"]
] = pd.DataFrame(
    calendar_values,
    index=analysis_data.index,
)

analysis_data["age_group"] = analysis_data["BAGE"]
analysis_data["household_income_band"] = analysis_data["BINC1"]
analysis_data["household_size"] = analysis_data["BFAM"]
analysis_data["marital_status"] = analysis_data["BMAR"]
analysis_data["household_with_child"] = (
    pd.to_numeric(analysis_data["DQ3A"], errors="coerce")
    .gt(0)
    .astype("int8")
)

In [16]:
party_size = pd.to_numeric(analysis_data["A6"], errors="coerce")
child_count = pd.to_numeric(analysis_data["A6A"], errors="coerce").fillna(0)
companion_columns = [f"A6B_{number}" for number in range(1, 8)]
companions = numeric_frame(analysis_data, companion_columns).fillna(0)

analysis_data["party_size_band"] = pd.cut(
    party_size,
    bins=[0, 1, 2, 4, np.inf],
    labels=["1인", "2인", "3~4인", "5인 이상"],
).astype("object")
analysis_data["with_child"] = child_count.gt(0).astype("int8")
analysis_data["companion_type_count"] = companions.eq(1).sum(axis=1)
analysis_data["with_family_relative"] = companions[
    ["A6B_1", "A6B_2"]
].eq(1).any(axis=1).astype("int8")
analysis_data["with_friend"] = companions["A6B_3"].eq(1).astype("int8")

transport_columns = ["D_TRA_B7_1", "D_TRA_B7_2", "D_TRA_B7_3"]
transports = numeric_frame(analysis_data, transport_columns)
analysis_data["uses_car_or_rental"] = any_code(transports, {1, 8})
analysis_data["uses_public_transport"] = any_code(
    transports,
    {2, 3, 4, 5, 6, 7, 10},
)
analysis_data["uses_other_transport"] = any_code(
    transports,
    {9, 11, 12},
)
analysis_data["transport_mode_count"] = transports.nunique(axis=1)

In [17]:
assert not region_codes.duplicated(
    ["시도_코드", "시군구_코드"]
).any(), "지역코드 복합키가 중복되었습니다."
region_codes["spot_code"] = (
    region_codes["시도_코드"].astype(int) * 1000
    + region_codes["시군구_코드"].astype(int)
)
assert region_codes["spot_code"].is_unique

spot_columns = [
    f"D_TRA_{number}_SPOT"
    for number in range(1, 18)
    if f"D_TRA_{number}_SPOT" in analysis_data.columns
]
lodging_columns = [
    f"D_TRA_{number}_Q6"
    for number in range(1, 18)
    if f"D_TRA_{number}_Q6" in analysis_data.columns
]
spots = numeric_frame(analysis_data, spot_columns)
lodging = numeric_frame(analysis_data, lodging_columns)
spot_long = (
    spots.assign(row_key=analysis_data.index)
    .melt(id_vars="row_key", value_name="spot_code")
    .dropna(subset=["spot_code"])
)
spot_long["spot_code"] = spot_long["spot_code"].astype(int)
spot_long = spot_long.merge(
    region_codes[["spot_code", "시도_코드", "시도명"]],
    on="spot_code",
    how="left",
    validate="many_to_one",
    indicator=True,
)
unmatched_spots = (
    spot_long.loc[spot_long["_merge"].eq("left_only"), ["spot_code"]]
    .value_counts()
    .reset_index(name="방문건수")
)
visited_sido = (
    spot_long.dropna(subset=["시도_코드"])
    .groupby("row_key")["시도_코드"]
    .agg(lambda values: {int(value) for value in values})
)

In [18]:
residence_to_sido = {
    1: 11,
    2: 21,
    3: 22,
    4: 23,
    5: 24,
    6: 25,
    7: 26,
    8: 29,
    9: 31,
    10: 32,
    11: 33,
    12: 34,
    13: 35,
    14: 36,
    15: 37,
    16: 38,
    17: 39,
}
residence_sido = pd.to_numeric(
    analysis_data["BARA"],
    errors="coerce",
).map(residence_to_sido)
capital_sido_codes = {11, 23, 31}


def visit_scope(row_key: int, residence: float) -> str:
    '''거주 시도와 방문 시도의 일치 범위를 분류한다.

    Args:
        row_key: 분석 데이터 행 인덱스.
        residence: 거주 시도 코드.

    Returns:
        관내, 관외, 혼합 또는 정보없음.
    '''
    codes = visited_sido.get(row_key, set())
    if not codes or pd.isna(residence):
        return "정보없음"
    matched = int(residence) in codes
    if matched and len(codes) == 1:
        return "관내"
    if matched:
        return "관내·관외 혼합"
    return "관외"


def capital_scope(row_key: int) -> str:
    '''방문 시도의 수도권 범위를 분류한다.

    Args:
        row_key: 분석 데이터 행 인덱스.

    Returns:
        수도권, 비수도권, 혼합 또는 정보없음.
    '''
    codes = visited_sido.get(row_key, set())
    if not codes:
        return "정보없음"
    if codes <= capital_sido_codes:
        return "수도권"
    if codes.isdisjoint(capital_sido_codes):
        return "비수도권"
    return "수도권·비수도권 혼합"


analysis_data["visit_scope"] = [
    visit_scope(row_key, residence)
    for row_key, residence in zip(
        analysis_data.index,
        residence_sido,
    )
]
analysis_data["capital_visit_pattern"] = [
    capital_scope(row_key) for row_key in analysis_data.index
]
analysis_data["visit_region_count"] = (
    analysis_data.index.map(
        spot_long.groupby("row_key")["spot_code"].nunique()
    )
    .to_series(index=analysis_data.index)
    .fillna(0)
    .to_numpy()
)
analysis_data["visit_region_scope"] = np.where(
    analysis_data["visit_region_count"].le(1),
    "단일지역",
    "다지역",
)

In [19]:
commercial_lodging_codes = set(range(1, 12))


def lodging_summary(row: pd.Series) -> str:
    '''숙박시설 코드 집합을 의미 중심 유형으로 요약한다.

    Args:
        row: 한 여행의 방문지별 숙박시설 코드.

    Returns:
        가족·친지집만, 상업숙박만, 혼합, 기타·무박 중 하나.
    '''
    codes = {int(value) for value in row.dropna()}
    has_family_home = 12 in codes
    has_commercial = bool(codes & commercial_lodging_codes)
    if has_family_home and not has_commercial and codes <= {12, 13}:
        return "가족·친지집만"
    if has_commercial and not has_family_home:
        return "상업숙박만"
    if has_family_home and has_commercial:
        return "혼합"
    return "기타·무박"


analysis_data["lodging_summary"] = lodging.apply(
    lodging_summary,
    axis=1,
)
analysis_data["family_home_only"] = analysis_data[
    "lodging_summary"
].eq("가족·친지집만").astype("int8")

reservation_columns = [
    f"A2_{number}"
    for number in [1, 2, 3, 4, 5, 6, 7, 10, 11]
]
reservations = numeric_frame(
    analysis_data,
    reservation_columns,
).fillna(0)
positive_reservations = reservations.drop(columns=["A2_11"])
analysis_data["has_reservation"] = any_selected(positive_reservations)
analysis_data["reservation_count"] = positive_reservations.eq(1).sum(axis=1)
analysis_data["reserved_transport"] = (
    reservations[["A2_3", "A2_4"]].eq(1).any(axis=1).astype("int8")
)
analysis_data["reserved_lodging"] = reservations["A2_1"].eq(1).astype("int8")
analysis_data["reserved_activity"] = (
    reservations[["A2_2", "A2_5", "A2_6", "A2_7"]]
    .eq(1)
    .any(axis=1)
    .astype("int8")
)

In [20]:
information_columns = ["A5_1", "A5_2", "A5_3"]
information = numeric_frame(analysis_data, information_columns)
analysis_data["information_none"] = any_code(information, {8})
analysis_data["information_online"] = any_code(information, {1})
analysis_data["information_people"] = any_code(information, {5})
analysis_data["information_source_count"] = information.nunique(axis=1)

activity_columns = [
    f"A4_{number}" for number in range(1, 22) if number != 18
]
activities = numeric_frame(analysis_data, activity_columns).fillna(0)
activity_groups = {
    "activity_nature": [1, 6],
    "activity_food": [2],
    "activity_culture": [4, 9, 13, 16],
    "activity_experience": [5, 12, 15],
    "activity_leisure": [3, 10, 14, 17],
    "activity_wellness": [7],
    "activity_shopping": [8],
    "activity_event": [11],
}
for feature_name, activity_codes in activity_groups.items():
    group_columns = [
        f"A4_{number}"
        for number in activity_codes
        if f"A4_{number}" in activities.columns
    ]
    analysis_data[feature_name] = any_selected(
        activities[group_columns]
    )
analysis_data["activity_group_count"] = analysis_data[
    list(activity_groups)
].sum(axis=1)

In [21]:
spend_category_columns = [
    f"A8{letter}" for letter in "ABCDEFGHIJ"
]
spend_categories = numeric_frame(
    analysis_data,
    spend_category_columns,
).clip(lower=0)
destination_total = pd.to_numeric(
    analysis_data["A8"],
    errors="coerce",
).replace(0, np.nan)
base_spend_features = []
for column in spend_category_columns:
    suffix = column.lower()
    indicator_name = f"spent_{suffix}"
    share_name = f"share_{suffix}"
    analysis_data[indicator_name] = (
        spend_categories[column].fillna(0).gt(0).astype("int8")
    )
    analysis_data[share_name] = (
        spend_categories[column] / destination_total
    ).clip(lower=0, upper=1)
    base_spend_features.extend([indicator_name, share_name])

spend_per_person = pd.to_numeric(
    analysis_data["D_TRA_ONE_COST"],
    errors="coerce",
).clip(lower=0)
spend_per_person_day = spend_per_person / analysis_data[
    "trip_days"
].replace(0, np.nan)
analysis_data["spend_per_person_day"] = spend_per_person_day
analysis_data["log_spend_per_person_day"] = spend_per_person_day

## 6. Full 세부 피처와 피처 계약

Full은 Base 전체에 세부 이동 순위, 방문 시도, 숙박 코드, 예약 항목,
`A4_18`을 제외한 활동, 정보원·사이트, 방문지 선택 이유·순위,
동반자 세부유형, 항목별 지출액·구성비를 추가한다. `P*` 인원 변수,
자유응답, 결과 문항, 직접 타깃·CASE·CHECK, 원 거주 시도는 강제
제외한다.

In [22]:
transport_rank_columns = [
    f"D_TRA_B7_RANKW_{number}" for number in range(1, 13)
]
visit_sido_features = []
for sido_code in sorted(region_codes["시도_코드"].unique()):
    feature_name = f"visit_sido_{int(sido_code)}"
    visited_rows = set(
        spot_long.loc[
            spot_long["시도_코드"].eq(sido_code),
            "row_key",
        ]
    )
    analysis_data[feature_name] = (
        analysis_data.index.isin(visited_rows).astype("int8")
    )
    visit_sido_features.append(feature_name)

lodging_detail_features = []
for lodging_code in range(1, 15):
    feature_name = f"has_lodging_{lodging_code}"
    analysis_data[feature_name] = (
        lodging.eq(lodging_code).any(axis=1).astype("int8")
    )
    lodging_detail_features.append(feature_name)

website_columns = ["A5A_1", "A5A_2", "A5A_3"]
reason_columns = ["A3_1", "A3_2", "A3_3"]
reason_weight_columns = [
    f"A3_RANKW_{number}" for number in range(1, 16)
]
companion_detail_features = companion_columns

/tmp/ipykernel_1189/921385914.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  analysis_data[feature_name] = (
/tmp/ipykernel_1189/921385914.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  analysis_data[feature_name] = (
/tmp/ipykernel_1189/921385914.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fram

In [23]:
detail_spend_pattern = re.compile(
    r"^(A7_\d+|A8[A-J]_\d+|D1_\d+)$"
)
detail_spend_sources = [
    column
    for column in analysis_data.columns
    if detail_spend_pattern.match(column)
]
full_spend_features = []
trip_total = pd.to_numeric(
    analysis_data["D_TRA_COST"],
    errors="coerce",
).replace(0, np.nan)
for source_column in detail_spend_sources:
    amount_feature = f"amount_{source_column.lower()}"
    share_feature = f"share_{source_column.lower()}"
    amount_values = pd.to_numeric(
        analysis_data[source_column],
        errors="coerce",
    ).clip(lower=0)
    analysis_data[amount_feature] = amount_values
    analysis_data[share_feature] = (
        amount_values / trip_total
    ).clip(lower=0, upper=1)
    full_spend_features.extend([amount_feature, share_feature])

/tmp/ipykernel_1189/3836601029.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  analysis_data[amount_feature] = amount_values
/tmp/ipykernel_1189/3836601029.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  analysis_data[share_feature] = (
/tmp/ipykernel_1189/3836601029.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, us

In [24]:
base_features = [
    "age_group",
    "household_income_band",
    "household_size",
    "marital_status",
    "household_with_child",
    "season",
    "overnight_type",
    "trip_days_band",
    "includes_weekday",
    "includes_weekend",
    "includes_holiday",
    "party_size_band",
    "with_child",
    "companion_type_count",
    "with_family_relative",
    "with_friend",
    "uses_car_or_rental",
    "uses_public_transport",
    "uses_other_transport",
    "transport_mode_count",
    "visit_scope",
    "capital_visit_pattern",
    "visit_region_scope",
    "lodging_summary",
    "family_home_only",
    "has_reservation",
    "reservation_count",
    "reserved_transport",
    "reserved_lodging",
    "reserved_activity",
    "information_none",
    "information_online",
    "information_people",
    "information_source_count",
    *activity_groups.keys(),
    "activity_group_count",
    "spend_per_person_day",
    "log_spend_per_person_day",
    *base_spend_features,
]
full_additional_features = [
    *transport_columns,
    *transport_rank_columns,
    *visit_sido_features,
    *lodging_detail_features,
    *reservation_columns,
    *activity_columns,
    *information_columns,
    *website_columns,
    *reason_columns,
    *reason_weight_columns,
    *companion_detail_features,
    *full_spend_features,
]
full_features = list(
    dict.fromkeys(base_features + full_additional_features)
)

forbidden_pattern = re.compile(
    r"(^YEAR$|^ID$|CASE|CHECK|^target$|^SA1_\d+$|^BARA$|"
    r"^P[A-Z0-9_]*$|^T[A-Z0-9_]*$|^A4_18$|"
    r"^A9(?:A_\d+)?$|^A10$|^A11$|^A12_\d+$)"
)
forbidden_features = [
    feature
    for feature in full_features
    if forbidden_pattern.search(feature)
]
assert not forbidden_features, (
    f"누출·강제 제외 변수가 모델 입력에 남았습니다: {forbidden_features}"
)
assert set(base_features) <= set(full_features)
assert len(base_features) == len(set(base_features))
assert len(full_features) == len(set(full_features))
missing_features = sorted(
    set(full_features) - set(analysis_data.columns)
)
assert not missing_features, f"생성되지 않은 피처: {missing_features}"

In [25]:
source_labels = codebook.set_index("column_name")[
    "column_label"
].to_dict()
categorical_base = {
    "age_group",
    "household_income_band",
    "household_size",
    "marital_status",
    "season",
    "overnight_type",
    "trip_days_band",
    "party_size_band",
    "visit_scope",
    "capital_visit_pattern",
    "visit_region_scope",
    "lodging_summary",
}
nominal_originals = set(
    codebook.loc[
        codebook["column_type"].eq("nominal"),
        "column_name",
    ]
)
domain_rules = [
    ("여행자", re.compile(r"age|household|marital")),
    ("일정", re.compile(r"season|overnight|trip_days|includes_")),
    ("동반", re.compile(r"party|child|companion|family|friend|A6B")),
    ("이동", re.compile(r"transport|D_TRA_B7")),
    ("방문지역", re.compile(r"visit_|capital_")),
    ("숙박", re.compile(r"lodging")),
    ("예약", re.compile(r"reserv|A2_")),
    ("정보", re.compile(r"information|A5")),
    ("활동", re.compile(r"activity|A4_")),
    ("소비", re.compile(r"spend|amount_|share_|spent_")),
]


def feature_domain(feature: str) -> str:
    '''피처 이름을 분석 도메인에 배정한다.

    Args:
        feature: 피처 이름.

    Returns:
        분석 도메인 이름.
    '''
    for domain, pattern in domain_rules:
        if pattern.search(feature):
            return domain
    return "기타"


def source_feature(feature: str) -> str:
    '''파생 피처의 대표 원변수를 기록한다.

    Args:
        feature: 피처 이름.

    Returns:
        원변수 또는 파생 규칙 설명.
    '''
    if feature in analysis_data.columns and feature in source_labels:
        return feature
    if feature.startswith(("amount_", "share_")):
        candidate = feature.split("_", 1)[1].upper()
        if candidate in source_labels:
            return candidate
    return "행 단위 파생"


dictionary_rows = []
for feature in full_features:
    source = source_feature(feature)
    dictionary_rows.append(
        {
            "feature": feature,
            "tier": "Base" if feature in base_features else "Full 추가",
            "domain": feature_domain(feature),
            "source": source,
            "label": source_labels.get(source, feature),
            "categorical": (
                feature in categorical_base
                or feature in nominal_originals
            ),
            "model_input": True,
        }
    )
feature_dictionary = pd.DataFrame(dictionary_rows)

In [26]:
selected_originals = set(
    feature_dictionary.loc[
        feature_dictionary["source"].isin(raw_data.columns),
        "source",
    ]
)
all_missing_set = set(empty_after_filter)


def exclusion_reason(column: str) -> str:
    '''원변수의 미사용 사유를 분류한다.

    Args:
        column: 원변수 이름.

    Returns:
        미사용 사유.
    '''
    if column in {"YEAR", "ID"}:
        return "키"
    if column == "WT_DOM":
        return "품질 확인 메타데이터(학습·분할·평가 미사용)"
    if column in selected_originals:
        return "Full 모델 원천 피처"
    if column in all_missing_set:
        return "CASE 1·2에서 전부 결측"
    if re.search(r"CASE|CHECK|^SA1_\d+$", column):
        return "타깃·여행유형 누출"
    if column == "BARA":
        return "원 거주 시도 강제 제외(관내·관외 파생에만 사용)"
    if re.search(
        r"(SYEAR|SMONTH|SDAY|EYEAR|EMONTH|EDAY)$",
        column,
    ):
        return "세부 날짜 강제 제외"
    if re.match(r"^P[A-Z0-9_]*$", column):
        return "비용 포함 인원 강제 제외"
    if re.match(r"^T[A-Z0-9_]*$", column):
        return "자유응답 텍스트 강제 제외"
    if column == "A4_18":
        return "가족·친지·친구 방문 직접신호 강제 제외"
    if re.match(r"^A9(?:A_\d+)?$|^A10$|^A11$|^A12_\d+$", column):
        return "만족도·재방문·추천·여행효과 결과 변수"
    if re.search(r"TOTAL$|^D_TRA_COST$|^A8$|^NA8$", column):
        return "세부 지출과 의미 중복인 총액"
    return "사전 피처 계약 미선정"


exclusion_log = pd.DataFrame(
    {
        "column_name": raw_data.columns,
        "reason": [
            exclusion_reason(column) for column in raw_data.columns
        ],
    }
)

## 7. 공통 Train/Validation/Test 분할과 전처리 파일 저장

`YEAR × target` 조합을 층화해 Train 70%, Validation 15%, Test 15%로
한 번만 나눈다. Base와 Full은 같은 `split_assignment.csv`를 사용한다.
키·타깃·`WT_DOM`은 Parquet에 보존하지만 모델 입력 목록과 분리한다.

In [27]:
key_columns = ["YEAR", "ID"]
metadata_columns = [*key_columns, "target", "WT_DOM"]
stratification_label = (
    analysis_data["YEAR"].astype(str)
    + "_"
    + analysis_data["target"].astype(str)
)
train_validation_index, test_index = train_test_split(
    analysis_data.index,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=stratification_label,
)
train_index, validation_index = train_test_split(
    train_validation_index,
    test_size=0.15 / 0.85,
    random_state=RANDOM_STATE + 1,
    stratify=stratification_label.loc[train_validation_index],
)
split_map = pd.Series("train", index=train_index)
split_map = pd.concat(
    [
        split_map,
        pd.Series("validation", index=validation_index),
        pd.Series("test", index=test_index),
    ]
)
assert split_map.index.is_unique
assert set(split_map.index) == set(analysis_data.index)

split_assignment = analysis_data[metadata_columns].copy()
split_assignment["split"] = split_assignment.index.map(split_map)
split_assignment = split_assignment[
    ["YEAR", "ID", "target", "WT_DOM", "split"]
]

split_distribution = (
    split_assignment.groupby(
        ["split", "YEAR", "target"],
        observed=True,
    )
    .size()
    .rename("N")
    .reset_index()
)
display(split_distribution)

,split,YEAR,target,N
0,test,2023,0,2917
1,test,2023,1,665
2,test,2024,0,2838
3,test,2024,1,681
4,test,2025,0,2931
5,test,2025,1,573
6,train,2023,0,13612
7,train,2023,1,3102
8,train,2024,0,13242
9,train,2024,1,3177


In [28]:
derived_base_data = analysis_data[
    metadata_columns + base_features
].copy()
derived_full_data = analysis_data[
    metadata_columns + full_features
].copy()


def load_complete_preprocessing() -> tuple[
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
    pd.DataFrame,
] | None:
    '''완전한 기존 전처리 산출물을 다시 읽는다.

    Returns:
        Base, Full, 피처 사전, 분할표. 파일이 없거나 계약이 불완전하면
        ``None``.
    '''
    if not all(path.is_file() for path in PREPROCESSING_ARTIFACT_PATHS):
        return None
    try:
        cached_base = pd.read_parquet(BASE_PARQUET_PATH)
        cached_full = pd.read_parquet(FULL_PARQUET_PATH)
        cached_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)
        cached_split = pd.read_csv(SPLIT_ASSIGNMENT_PATH)
        pd.read_csv(EXCLUSION_LOG_PATH, nrows=1)
        assert len(cached_base) == EXPECTED_ANALYSIS_N
        assert len(cached_full) == EXPECTED_ANALYSIS_N
        assert cached_base[key_columns].equals(
            cached_full[key_columns]
        )
        assert cached_base["target"].equals(cached_full["target"])
        assert set(metadata_columns + base_features) <= set(
            cached_base.columns
        )
        assert set(metadata_columns + full_features) <= set(
            cached_full.columns
        )
        assert set(cached_dictionary["feature"]) == set(full_features)
        assert cached_split[key_columns].equals(
            cached_base[key_columns]
        )
        assert set(cached_split["split"]) == {
            "train",
            "validation",
            "test",
        }
    except (AssertionError, OSError, ValueError):
        return None
    return (
        cached_base,
        cached_full,
        cached_dictionary,
        cached_split,
    )


cached_preprocessing = load_complete_preprocessing()
if cached_preprocessing is None:
    base_data = derived_base_data
    full_data = derived_full_data
    CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)
    atomic_parquet(base_data, BASE_PARQUET_PATH)
    atomic_parquet(full_data, FULL_PARQUET_PATH)
    atomic_csv(feature_dictionary, FEATURE_DICTIONARY_PATH)
    atomic_csv(exclusion_log, EXCLUSION_LOG_PATH)
    atomic_csv(split_assignment, SPLIT_ASSIGNMENT_PATH)
    print(f"전처리 산출물 저장: {CLASSIFICATION_DIR}")
else:
    (
        base_data,
        full_data,
        feature_dictionary,
        split_assignment,
    ) = cached_preprocessing
    print(f"완전한 기존 전처리 산출물 재사용: {CLASSIFICATION_DIR}")

assert base_data[key_columns].equals(full_data[key_columns])
assert base_data["target"].equals(full_data["target"])
assert "WT_DOM" not in base_features
assert "WT_DOM" not in full_features
assert not base_data.duplicated(key_columns).any()
assert not full_data.duplicated(key_columns).any()

print(f"Base 피처: {len(base_features):,}개")
print(f"Full 피처: {len(full_features):,}개")

전처리 산출물 저장: /content/drive/MyDrive/tourism_poster/data/preprocess/classification
Base 피처: 65개
Full 피처: 299개


## 8. Train 전용 전처리 Pipeline

결측 대치, 원핫 인코딩, 스케일링과 1인·1일 지출의 99% 상한은
Pipeline 안에서 Train에만 적합한다. Validation/Test는 `transform`만
적용한다.

In [29]:
class SpendTransformer(BaseEstimator, TransformerMixin):
    '''1인·1일 지출의 Train 99% 상한과 로그 변환을 적용한다.'''

    def __init__(
        self,
        raw_column: str = "spend_per_person_day",
        log_column: str = "log_spend_per_person_day",
        quantile: float = 0.99,
    ) -> None:
        self.raw_column = raw_column
        self.log_column = log_column
        self.quantile = quantile
        self.upper_bound_: float | None = None

    def fit(
        self,
        X: pd.DataFrame,
        y: pd.Series | None = None,
    ) -> "SpendTransformer":
        '''Train에서 지출 상한을 학습한다.

        Args:
            X: 원 피처 DataFrame.
            y: 사용하지 않는 타깃.

        Returns:
            적합된 변환기.
        '''
        del y
        values = pd.to_numeric(
            X[self.raw_column],
            errors="coerce",
        )
        self.upper_bound_ = float(values.quantile(self.quantile))
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        '''학습한 상한으로 지출을 자르고 로그 피처를 만든다.

        Args:
            X: 변환할 피처 DataFrame.

        Returns:
            지출 피처가 변환된 DataFrame.
        '''
        if self.upper_bound_ is None:
            raise RuntimeError("SpendTransformer를 먼저 fit해야 합니다.")
        transformed = X.copy()
        capped = pd.to_numeric(
            transformed[self.raw_column],
            errors="coerce",
        ).clip(lower=0, upper=self.upper_bound_)
        transformed[self.raw_column] = capped
        transformed[self.log_column] = np.log1p(capped)
        return transformed


def make_one_hot_encoder() -> OneHotEncoder:
    '''설치된 scikit-learn에 맞는 OneHotEncoder를 만든다.

    Returns:
        미관측 범주를 무시하는 sparse OneHotEncoder.
    '''
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
        )


def build_pipeline(
    estimator: BaseEstimator,
    feature_names: list[str],
) -> Pipeline:
    '''Train 전용 전처리와 분류기를 묶은 Pipeline을 만든다.

    Args:
        estimator: 최종 분류기.
        feature_names: Base 또는 Full 피처 목록.

    Returns:
        지출 변환, 전처리, 분류기를 묶은 Pipeline.
    '''
    categorical_flags = (
        feature_dictionary["categorical"]
        .astype(str)
        .str.lower()
        .eq("true")
    )
    categorical = feature_dictionary.loc[
        feature_dictionary["feature"].isin(feature_names)
        & categorical_flags,
        "feature",
    ].tolist()
    numeric = [
        feature
        for feature in feature_names
        if feature not in categorical
    ]
    numeric_pipeline = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False)),
        ]
    )
    categorical_pipeline = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent",
                    keep_empty_features=True,
                ),
            ),
            ("onehot", make_one_hot_encoder()),
        ]
    )
    preprocessor = ColumnTransformer(
        [
            ("numeric", numeric_pipeline, numeric),
            (
                "categorical",
                categorical_pipeline,
                categorical,
            ),
        ],
        sparse_threshold=1.0,
    )
    return Pipeline(
        [
            ("spend", SpendTransformer()),
            ("preprocess", preprocessor),
            ("model", estimator),
        ]
    )

## 9. 모델 후보와 평가 함수

Base에서 Elastic Net, 랜덤포레스트, XGBoost, LightGBM, CatBoost를
동일한 `YEAR × target` 3-fold 층화 CV로 비교한다. 후보는 모델당
`SEARCH_CANDIDATES`개를 재현 가능한 난수로 추출한다.

In [30]:
train_mask = split_assignment["split"].eq("train")
validation_mask = split_assignment["split"].eq("validation")
test_mask = split_assignment["split"].eq("test")

y_train = base_data.loc[train_mask, "target"].astype(int)
y_validation = base_data.loc[validation_mask, "target"].astype(int)
y_test = base_data.loc[test_mask, "target"].astype(int)
train_cv_labels = (
    base_data.loc[train_mask, "YEAR"].astype(str)
    + "_"
    + y_train.astype(str)
)
positive_ratio = float(
    y_train.eq(0).sum() / y_train.eq(1).sum()
)


def model_specs() -> dict[str, dict]:
    '''비교 모델과 하이퍼파라미터 후보 공간을 만든다.

    Returns:
        모델 이름별 estimator와 parameter distribution.
    '''
    return {
        "elastic_net": {
            "estimator": LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                max_iter=3_000,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
            "params": {
                "model__C": np.logspace(-3, 2, 30),
                "model__l1_ratio": np.linspace(0.05, 0.95, 19),
                "model__class_weight": [None, "balanced"],
            },
        },
        "random_forest": {
            "estimator": RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
            "params": {
                "model__n_estimators": [300, 500, 700, 900],
                "model__max_depth": [None, 8, 12, 18, 24],
                "model__min_samples_leaf": [1, 2, 5, 10, 20],
                "model__max_features": ["sqrt", "log2", 0.5, 0.8],
                "model__class_weight": [None, "balanced"],
            },
        },
        "xgboost": {
            "estimator": XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
            "params": {
                "model__n_estimators": [300, 500, 700, 900],
                "model__max_depth": [3, 4, 5, 6, 8],
                "model__learning_rate": [0.02, 0.04, 0.06, 0.1],
                "model__subsample": [0.7, 0.85, 1.0],
                "model__colsample_bytree": [0.7, 0.85, 1.0],
                "model__min_child_weight": [1, 3, 5, 10],
                "model__reg_lambda": [0.5, 1.0, 3.0, 10.0],
                "model__scale_pos_weight": [
                    1.0,
                    positive_ratio,
                ],
            },
        },
        "lightgbm": {
            "estimator": LGBMClassifier(
                objective="binary",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=-1,
            ),
            "params": {
                "model__n_estimators": [300, 500, 700, 900],
                "model__num_leaves": [15, 31, 63, 127],
                "model__learning_rate": [0.02, 0.04, 0.06, 0.1],
                "model__subsample": [0.7, 0.85, 1.0],
                "model__colsample_bytree": [0.7, 0.85, 1.0],
                "model__min_child_samples": [10, 20, 40, 80],
                "model__reg_lambda": [0.0, 1.0, 5.0, 10.0],
                "model__class_weight": [None, "balanced"],
            },
        },
        "catboost": {
            "estimator": CatBoostClassifier(
                loss_function="Logloss",
                verbose=False,
                allow_writing_files=False,
                random_seed=RANDOM_STATE,
                thread_count=-1,
            ),
            "params": {
                "model__iterations": [300, 500, 700, 900],
                "model__depth": [4, 5, 6, 8, 10],
                "model__learning_rate": [0.02, 0.04, 0.06, 0.1],
                "model__l2_leaf_reg": [1.0, 3.0, 5.0, 10.0],
                "model__random_strength": [0.0, 0.5, 1.0, 2.0],
                "model__auto_class_weights": [None, "Balanced"],
            },
        },
    }


def binary_metrics(
    y_true: pd.Series | np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> dict[str, float]:
    '''이진 분류 평가지표를 계산한다.

    Args:
        y_true: 실제 타깃.
        probabilities: 양성 클래스 확률.
        threshold: 양성 판정 임계값.

    Returns:
        F1, 정밀도, 재현율, PR-AUC, ROC-AUC.
    '''
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    return {
        "f1": f1_score(y_true, predictions),
        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),
        "pr_auc": average_precision_score(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
    }


def select_threshold(
    y_true: pd.Series | np.ndarray,
    probabilities: np.ndarray,
) -> tuple[float, pd.DataFrame]:
    '''Validation F1이 최대인 임계값을 고른다.

    동률이면 0.5와 가까운 임계값을 사용한다.

    Args:
        y_true: 실제 Validation 타깃.
        probabilities: 양성 클래스 확률.

    Returns:
        선택 임계값과 전체 임계값 평가표.
    '''
    thresholds = np.round(np.arange(0.05, 0.9501, 0.005), 3)
    rows = []
    for threshold in thresholds:
        metrics = binary_metrics(y_true, probabilities, threshold)
        rows.append({"threshold": threshold, **metrics})
    table = pd.DataFrame(rows)
    table["distance_to_0_5"] = (
        table["threshold"] - 0.5
    ).abs()
    best = table.sort_values(
        ["f1", "distance_to_0_5"],
        ascending=[False, True],
    ).iloc[0]
    return float(best["threshold"]), table.drop(
        columns="distance_to_0_5"
    )

## 10. Base 하이퍼파라미터 탐색

fold 하나가 끝날 때마다 진행표와 상태 JSON을 저장한다. 재개 시 이미
완료된 `(모델, 후보, fold)`를 건너뛴다. CV 선택 지표는 0.5 임계값의
평균 F1이며, 탐색 종료 후 Train 적합 모델의 Validation 임계값을 별도로
고른다.

In [38]:
SEARCH_PROGRESS_PATH = MODELS_DIR / "base_search_progress.csv"
SEARCH_STATE_PATH = MODELS_DIR / "base_search_state.json"


def load_search_progress() -> pd.DataFrame:
    '''저장된 Base 탐색 진행표를 읽는다.

    Returns:
        진행표가 없으면 빈 DataFrame.
    '''
    if SEARCH_PROGRESS_PATH.is_file():
        return pd.read_csv(SEARCH_PROGRESS_PATH)
    return pd.DataFrame()


def search_base_model(
    model_name: str,
    specification: dict,
    X_train: pd.DataFrame,
    y_train_values: pd.Series,
    cv_labels: pd.Series,
) -> tuple[dict, pd.DataFrame]:
    '''한 Base 모델의 후보를 공통 층화 CV로 탐색한다.

    Args:
        model_name: 모델 식별자.
        specification: estimator와 파라미터 공간.
        X_train: Base Train 피처.
        y_train_values: Train 타깃.
        cv_labels: YEAR × target 층화 라벨.

    Returns:
        최적 파라미터와 후보별 CV 요약표.
    '''
    candidates = list(
        ParameterSampler(
            specification["params"],
            n_iter=SEARCH_CANDIDATES,
            random_state=RANDOM_STATE,
        )
    )
    splitter = StratifiedKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    progress = load_search_progress()
    completed = set()
    if not progress.empty:
        completed = set(
            zip(
                progress["model"],
                progress["candidate"].astype(int),
                progress["fold"].astype(int),
            )
        )

    for candidate_index, parameters in enumerate(candidates):
        for fold_index, (fit_positions, score_positions) in enumerate(
            splitter.split(X_train, cv_labels),
            start=1,
        ):
            key = (model_name, candidate_index, fold_index)
            if key in completed:
                continue
            pipeline = build_pipeline(
                clone(specification["estimator"]),
                base_features,
            )
            clean_parameters = {
                key: value
                for key, value in parameters.items()
                if not (
                    key.endswith("auto_class_weights")
                    and value is None
                )
            }
            pipeline.set_params(**clean_parameters)
            pipeline.fit(
                X_train.iloc[fit_positions],
                y_train_values.iloc[fit_positions],
            )
            probabilities = pipeline.predict_proba(
                X_train.iloc[score_positions]
            )[:, 1]
            metrics = binary_metrics(
                y_train_values.iloc[score_positions],
                probabilities,
                threshold=0.5,
            )
            new_row = pd.DataFrame(
                [
                    {
                        "model": model_name,
                        "candidate": candidate_index,
                        "fold": fold_index,
                        "parameters": json.dumps(
                            parameters,
                            ensure_ascii=False,
                            default=float,
                            sort_keys=True,
                        ),
                        **metrics,
                    }
                ]
            )
            progress = pd.concat(
                [progress, new_row],
                ignore_index=True,
            )
            atomic_csv(progress, SEARCH_PROGRESS_PATH)
            atomic_json(
                {
                    "last_completed": {
                        "model": model_name,
                        "candidate": candidate_index,
                        "fold": fold_index,
                    },
                    "completed_folds": len(progress),
                },
                SEARCH_STATE_PATH,
            )
            completed.add(key)

    model_progress = progress.loc[
        progress["model"].eq(model_name)
    ].copy()
    summary = (
        model_progress.groupby(
            ["candidate", "parameters"],
            as_index=False,
        )
        .agg(
            mean_f1=("f1", "mean"),
            mean_pr_auc=("pr_auc", "mean"),
            mean_roc_auc=("roc_auc", "mean"),
            completed_folds=("fold", "nunique"),
        )
    )
    assert summary["completed_folds"].eq(CV_FOLDS).all()
    summary = summary.sort_values(
        ["mean_f1", "mean_pr_auc"],
        ascending=False,
    )
    best_parameters = json.loads(summary.iloc[0]["parameters"])
    return best_parameters, summary

In [32]:
specifications = model_specs()
X_base_train = base_data.loc[train_mask, base_features]
X_base_validation = base_data.loc[
    validation_mask,
    base_features,
]
base_best_parameters = {}
base_validation_rows = []

for model_name, specification in specifications.items():
    parameters_path = (
        MODELS_DIR / f"base_{model_name}_best_parameters.json"
    )
    pipeline_path = (
        MODELS_DIR / f"base_{model_name}_train_pipeline.joblib"
    )
    prediction_path = (
        MODELS_DIR / f"base_{model_name}_validation_predictions.csv"
    )
    if (
        parameters_path.is_file()
        and pipeline_path.is_file()
        and prediction_path.is_file()
    ):
        best_parameters = json.loads(
            parameters_path.read_text(encoding="utf-8")
        )
        fitted_pipeline = joblib.load(pipeline_path)
        prediction_table = pd.read_csv(prediction_path)
        validation_probabilities = prediction_table[
            "probability"
        ].to_numpy()
        threshold = float(prediction_table["threshold"].iloc[0])
    else:
        best_parameters, search_summary = search_base_model(
            model_name,
            specification,
            X_base_train,
            y_train,
            train_cv_labels,
        )
        atomic_csv(
            search_summary,
            TABLES_DIR / f"base_{model_name}_search_summary.csv",
        )
        fitted_pipeline = build_pipeline(
            clone(specification["estimator"]),
            base_features,
        )
        # clean_best_parameters = {
        #     key: value
        #     for key, value in best_parameters.items()
        #     if not (
        #         key.endswith("auto_class_weights")
        #         and value is None
        #     )
        # }
        # fitted_pipeline.set_params(**clean_best_parameters)
        fitted_pipeline.set_params(**best_parameters)
        fitted_pipeline.fit(X_base_train, y_train)
        validation_probabilities = fitted_pipeline.predict_proba(
            X_base_validation
        )[:, 1]
        threshold, threshold_table = select_threshold(
            y_validation,
            validation_probabilities,
        )
        atomic_csv(
            threshold_table,
            TABLES_DIR
            / f"base_{model_name}_validation_thresholds.csv",
        )
        prediction_table = split_assignment.loc[
            validation_mask,
            ["YEAR", "ID", "target"],
        ].copy()
        prediction_table["probability"] = validation_probabilities
        prediction_table["threshold"] = threshold
        atomic_json(best_parameters, parameters_path)
        atomic_joblib(fitted_pipeline, pipeline_path)
        atomic_csv(prediction_table, prediction_path)

    base_best_parameters[model_name] = best_parameters
    metrics = binary_metrics(
        y_validation,
        validation_probabilities,
        threshold,
    )
    base_validation_rows.append(
        {
            "tier": "Base",
            "model": model_name,
            "threshold": threshold,
            **metrics,
        }
    )
    print(
        f"Base {model_name} 완료: "
        f"Validation F1={metrics['f1']:.4f}"
    )

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which 

Base elastic_net 완료: Validation F1=0.6266
Base random_forest 완료: Validation F1=0.6338
Base xgboost 완료: Validation F1=0.6487
Base lightgbm 완료: Validation F1=0.6479


CatBoostError: catboost/private/libs/options/json_helper.h:41: Can't parse parameter "auto_class_weights" with value: null

### catboost 별도 실행

In [33]:
for model_name in [
    "elastic_net",
    "random_forest",
    "xgboost",
    "lightgbm",
]:
    parameters_path = (
        MODELS_DIR / f"base_{model_name}_best_parameters.json"
    )
    pipeline_path = (
        MODELS_DIR / f"base_{model_name}_train_pipeline.joblib"
    )
    prediction_path = (
        MODELS_DIR / f"base_{model_name}_validation_predictions.csv"
    )

    print(
        model_name,
        parameters_path.is_file(),
        pipeline_path.is_file(),
        prediction_path.is_file(),
    )

elastic_net True True True
random_forest True True True
xgboost True True True
lightgbm True True True


In [39]:
model_name = "catboost"
specification = model_specs()[model_name]

X_base_train = base_data.loc[
    train_mask,
    base_features,
]
X_base_validation = base_data.loc[
    validation_mask,
    base_features,
]

parameters_path = (
    MODELS_DIR / f"base_{model_name}_best_parameters.json"
)
pipeline_path = (
    MODELS_DIR / f"base_{model_name}_train_pipeline.joblib"
)
prediction_path = (
    MODELS_DIR / f"base_{model_name}_validation_predictions.csv"
)

best_parameters, search_summary = search_base_model(
    model_name,
    specification,
    X_base_train,
    y_train,
    train_cv_labels,
)

atomic_csv(
    search_summary,
    TABLES_DIR / f"base_{model_name}_search_summary.csv",
)

fitted_pipeline = build_pipeline(
    clone(specification["estimator"]),
    base_features,
)

clean_best_parameters = {
    key: value
    for key, value in best_parameters.items()
    if not (
        key.endswith("auto_class_weights")
        and value is None
    )
}

fitted_pipeline.set_params(**clean_best_parameters)
fitted_pipeline.fit(X_base_train, y_train)

validation_probabilities = fitted_pipeline.predict_proba(
    X_base_validation
)[:, 1]

threshold, threshold_table = select_threshold(
    y_validation,
    validation_probabilities,
)

atomic_csv(
    threshold_table,
    TABLES_DIR
    / f"base_{model_name}_validation_thresholds.csv",
)

prediction_table = split_assignment.loc[
    validation_mask,
    ["YEAR", "ID", "target"],
].copy()

prediction_table["probability"] = validation_probabilities
prediction_table["threshold"] = threshold

atomic_json(
    clean_best_parameters,
    parameters_path,
)
atomic_joblib(
    fitted_pipeline,
    pipeline_path,
)
atomic_csv(
    prediction_table,
    prediction_path,
)

metrics = binary_metrics(
    y_validation,
    validation_probabilities,
    threshold,
)

print(
    f"Base {model_name} 완료: "
    f"Validation F1={metrics['f1']:.4f}"
)

Base catboost 완료: Validation F1=0.6433


In [40]:
specifications = model_specs()

base_best_parameters = {}
base_validation_rows = []

for model_name in specifications:
    parameters_path = (
        MODELS_DIR / f"base_{model_name}_best_parameters.json"
    )
    pipeline_path = (
        MODELS_DIR / f"base_{model_name}_train_pipeline.joblib"
    )
    prediction_path = (
        MODELS_DIR / f"base_{model_name}_validation_predictions.csv"
    )

    required_paths = [
        parameters_path,
        pipeline_path,
        prediction_path,
    ]

    missing_paths = [
        path
        for path in required_paths
        if not path.is_file()
    ]

    if missing_paths:
        missing_text = "\n".join(
            str(path)
            for path in missing_paths
        )
        raise FileNotFoundError(
            f"{model_name}의 저장 파일이 부족합니다.\n"
            f"{missing_text}"
        )

    best_parameters = json.loads(
        parameters_path.read_text(encoding="utf-8")
    )
    prediction_table = pd.read_csv(prediction_path)

    validation_probabilities = prediction_table[
        "probability"
    ].to_numpy()

    threshold = float(
        prediction_table["threshold"].iloc[0]
    )

    metrics = binary_metrics(
        y_validation,
        validation_probabilities,
        threshold,
    )

    base_best_parameters[model_name] = best_parameters
    base_validation_rows.append(
        {
            "tier": "Base",
            "model": model_name,
            "threshold": threshold,
            **metrics,
        }
    )

    print(
        f"Base {model_name} 불러오기 완료: "
        f"Validation F1={metrics['f1']:.4f}"
    )

base_validation_table = pd.DataFrame(
    base_validation_rows
)

base_validation_table

Base elastic_net 불러오기 완료: Validation F1=0.6266
Base random_forest 불러오기 완료: Validation F1=0.6338
Base xgboost 불러오기 완료: Validation F1=0.6487
Base lightgbm 불러오기 완료: Validation F1=0.6479
Base catboost 불러오기 완료: Validation F1=0.6433


,tier,model,threshold,f1,precision,recall,pr_auc,roc_auc
0,Base,elastic_net,0.285,0.626596,0.778638,0.524231,0.666604,0.835084
1,Base,random_forest,0.445,0.633846,0.684179,0.590412,0.686615,0.848211
2,Base,xgboost,0.280,0.648679,0.712859,0.595102,0.699125,0.852390
3,Base,lightgbm,0.640,0.647921,0.760534,0.564356,0.697295,0.852821
4,Base,catboost,0.610,0.643314,0.774194,0.550287,0.686860,0.846160


## 11. Full 모델 적합과 Validation 비교

각 Full 모델은 같은 알고리즘의 Base 최적 하이퍼파라미터를 그대로
사용한다. Full에서는 별도 CV 탐색을 하지 않는다. 알고리즘 하나가
끝날 때마다 Train 적합 Pipeline, Validation 예측과 임계값을 저장한다.

In [41]:
X_full_train = full_data.loc[train_mask, full_features]
X_full_validation = full_data.loc[
    validation_mask,
    full_features,
]
full_validation_rows = []

for model_name, specification in specifications.items():
    pipeline_path = (
        MODELS_DIR / f"full_{model_name}_train_pipeline.joblib"
    )
    prediction_path = (
        MODELS_DIR / f"full_{model_name}_validation_predictions.csv"
    )
    if pipeline_path.is_file() and prediction_path.is_file():
        fitted_pipeline = joblib.load(pipeline_path)
        prediction_table = pd.read_csv(prediction_path)
        validation_probabilities = prediction_table[
            "probability"
        ].to_numpy()
        threshold = float(prediction_table["threshold"].iloc[0])
    else:
        fitted_pipeline = build_pipeline(
            clone(specification["estimator"]),
            full_features,
        )
        fitted_pipeline.set_params(
            **base_best_parameters[model_name]
        )
        fitted_pipeline.fit(X_full_train, y_train)
        validation_probabilities = fitted_pipeline.predict_proba(
            X_full_validation
        )[:, 1]
        threshold, threshold_table = select_threshold(
            y_validation,
            validation_probabilities,
        )
        atomic_csv(
            threshold_table,
            TABLES_DIR
            / f"full_{model_name}_validation_thresholds.csv",
        )
        prediction_table = split_assignment.loc[
            validation_mask,
            ["YEAR", "ID", "target"],
        ].copy()
        prediction_table["probability"] = validation_probabilities
        prediction_table["threshold"] = threshold
        atomic_joblib(fitted_pipeline, pipeline_path)
        atomic_csv(prediction_table, prediction_path)

    metrics = binary_metrics(
        y_validation,
        validation_probabilities,
        threshold,
    )
    full_validation_rows.append(
        {
            "tier": "Full",
            "model": model_name,
            "threshold": threshold,
            **metrics,
        }
    )
    print(
        f"Full {model_name} 완료: "
        f"Validation F1={metrics['f1']:.4f}"
    )

validation_leaderboard = pd.DataFrame(
    base_validation_rows + full_validation_rows
)
validation_leaderboard["tier_priority"] = (
    validation_leaderboard["tier"].ne("Base").astype(int)
)
validation_leaderboard = validation_leaderboard.sort_values(
    ["f1", "pr_auc", "tier_priority"],
    ascending=[False, False, True],
).drop(columns="tier_priority")
atomic_csv(
    validation_leaderboard,
    TABLES_DIR / "validation_leaderboard.csv",
)
display(validation_leaderboard)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Full elastic_net 완료: Validation F1=0.3310
Full random_forest 완료: Validation F1=0.6511
Full xgboost 완료: Validation F1=0.6841
Full lightgbm 완료: Validation F1=0.6878
Full catboost 완료: Validation F1=0.6804


,tier,model,threshold,f1,precision,recall,pr_auc,roc_auc
8,Full,lightgbm,0.615,0.687812,0.788022,0.610214,0.729890,0.873743
7,Full,xgboost,0.350,0.684148,0.798887,0.598228,0.729496,0.871261
9,Full,catboost,0.580,0.680382,0.766319,0.611777,0.729926,0.872404
6,Full,random_forest,0.505,0.651136,0.737954,0.582595,0.698927,0.858178
2,Base,xgboost,0.280,0.648679,0.712859,0.595102,0.699125,0.852390
3,Base,lightgbm,0.640,0.647921,0.760534,0.564356,0.697295,0.852821
4,Base,catboost,0.610,0.643314,0.774194,0.550287,0.686860,0.846160
1,Base,random_forest,0.445,0.633846,0.684179,0.590412,0.686615,0.848211
0,Base,elastic_net,0.285,0.626596,0.778638,0.524231,0.666604,0.835084
5,Full,elastic_net,0.180,0.330986,0.199131,0.979677,0.660382,0.832981


## 12. Base 대 Full paired bootstrap

각 tier의 Validation 최고 모델을 같은 Validation 행에서 비교한다.
2,000회 paired bootstrap으로 `Full − Base F1` 분포와 95% 구간을
저장한다.

In [42]:
best_base_row = validation_leaderboard.loc[
    validation_leaderboard["tier"].eq("Base")
].iloc[0]
best_full_row = validation_leaderboard.loc[
    validation_leaderboard["tier"].eq("Full")
].iloc[0]


def load_validation_prediction(
    tier: str,
    model_name: str,
) -> pd.DataFrame:
    '''저장된 Validation 예측표를 키 순서로 읽는다.

    Args:
        tier: Base 또는 Full.
        model_name: 모델 식별자.

    Returns:
        YEAR + ID로 정렬한 Validation 예측표.
    '''
    path = MODELS_DIR / (
        f"{tier.lower()}_{model_name}_validation_predictions.csv"
    )
    return pd.read_csv(path).sort_values(["YEAR", "ID"]).reset_index(
        drop=True
    )


best_base_predictions = load_validation_prediction(
    "Base",
    best_base_row["model"],
)
best_full_predictions = load_validation_prediction(
    "Full",
    best_full_row["model"],
)
assert best_base_predictions[["YEAR", "ID", "target"]].equals(
    best_full_predictions[["YEAR", "ID", "target"]]
)

rng = np.random.default_rng(RANDOM_STATE)
bootstrap_rows = []
validation_n = len(best_base_predictions)
for iteration in range(BOOTSTRAP_ITERATIONS):
    sampled_positions = rng.integers(
        0,
        validation_n,
        size=validation_n,
    )
    sampled_target = best_base_predictions["target"].to_numpy()[
        sampled_positions
    ]
    base_f1 = f1_score(
        sampled_target,
        (
            best_base_predictions["probability"].to_numpy()[
                sampled_positions
            ]
            >= best_base_row["threshold"]
        ),
    )
    full_f1 = f1_score(
        sampled_target,
        (
            best_full_predictions["probability"].to_numpy()[
                sampled_positions
            ]
            >= best_full_row["threshold"]
        ),
    )
    bootstrap_rows.append(
        {
            "iteration": iteration + 1,
            "base_f1": base_f1,
            "full_f1": full_f1,
            "full_minus_base_f1": full_f1 - base_f1,
        }
    )

bootstrap_results = pd.DataFrame(bootstrap_rows)
bootstrap_summary = pd.DataFrame(
    [
        {
            "base_model": best_base_row["model"],
            "full_model": best_full_row["model"],
            "iterations": BOOTSTRAP_ITERATIONS,
            "mean_difference": bootstrap_results[
                "full_minus_base_f1"
            ].mean(),
            "ci_2_5": bootstrap_results[
                "full_minus_base_f1"
            ].quantile(0.025),
            "ci_97_5": bootstrap_results[
                "full_minus_base_f1"
            ].quantile(0.975),
        }
    ]
)
atomic_csv(
    bootstrap_results,
    TABLES_DIR / "paired_bootstrap_distribution.csv",
)
atomic_csv(
    bootstrap_summary,
    TABLES_DIR / "paired_bootstrap_summary.csv",
)
display(bootstrap_summary)

,base_model,full_model,iterations,mean_difference,ci_2_5,ci_97_5
0,xgboost,lightgbm,2000,0.038933,0.02722,0.050784


## 13. 최종 구성 재학습과 Test 1회 평가

Validation F1 → PR-AUC → Base 우선 규칙으로 한 구성을 선택한다.
선택된 알고리즘·tier만 Train+Validation 85%로 재학습하고, Validation에서
고정한 임계값으로 Test를 한 번 평가한다.

In [43]:
selected_row = validation_leaderboard.iloc[0]
selected_tier = selected_row["tier"]
selected_model_name = selected_row["model"]
selected_threshold = float(selected_row["threshold"])
selected_features = (
    base_features if selected_tier == "Base" else full_features
)
selected_data = base_data if selected_tier == "Base" else full_data
train_validation_mask = split_assignment["split"].isin(
    ["train", "validation"]
)
X_train_validation = selected_data.loc[
    train_validation_mask,
    selected_features,
]
y_train_validation = selected_data.loc[
    train_validation_mask,
    "target",
].astype(int)
X_test = selected_data.loc[test_mask, selected_features]

final_pipeline = build_pipeline(
    clone(specifications[selected_model_name]["estimator"]),
    selected_features,
)
final_pipeline.set_params(
    **base_best_parameters[selected_model_name]
)
final_pipeline.fit(X_train_validation, y_train_validation)
test_probabilities = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = binary_metrics(
    y_test,
    test_probabilities,
    selected_threshold,
)
test_predictions = (
    test_probabilities >= selected_threshold
).astype(int)
test_confusion_matrix = confusion_matrix(
    y_test,
    test_predictions,
    labels=[0, 1],
)

final_prediction_table = split_assignment.loc[
    test_mask,
    ["YEAR", "ID", "target"],
].copy()
final_prediction_table["probability"] = test_probabilities
final_prediction_table["prediction"] = test_predictions
final_prediction_table["threshold"] = selected_threshold

final_summary = pd.DataFrame(
    [
        {
            "tier": selected_tier,
            "model": selected_model_name,
            "threshold": selected_threshold,
            **test_metrics,
            "test_N": len(y_test),
        }
    ]
)
atomic_joblib(final_pipeline, MODELS_DIR / "final_pipeline.joblib")
atomic_json(
    {
        "tier": selected_tier,
        "model": selected_model_name,
        "threshold": selected_threshold,
        "features": selected_features,
        "parameters": base_best_parameters[selected_model_name],
    },
    MODELS_DIR / "final_configuration.json",
)
atomic_csv(
    final_prediction_table,
    MODELS_DIR / "final_test_predictions.csv",
)
atomic_csv(final_summary, TABLES_DIR / "final_test_metrics.csv")
atomic_csv(
    pd.DataFrame(
        test_confusion_matrix,
        index=["실제 CASE 1", "실제 CASE 2"],
        columns=["예측 CASE 1", "예측 CASE 2"],
    ).reset_index(names="actual"),
    TABLES_DIR / "final_test_confusion_matrix.csv",
)
display(final_summary)

,tier,model,threshold,f1,precision,recall,pr_auc,roc_auc,test_N
0,Full,lightgbm,0.615,0.672388,0.778082,0.591975,0.72637,0.867407,10605


## 14. Base/Full SHAP

각 tier의 Validation 최고 모델을 동일한 `YEAR × target` 층화 표본으로
설명한다. 트리 모델은 `TreeExplainer`, Elastic Net은
`LinearExplainer`를 사용한다. 원핫 피처를 원변수로 합산해 도메인
중요도와 CASE 1·2 방향도 저장한다.

In [44]:
def stratified_sample_positions(
    frame: pd.DataFrame,
    sample_size: int,
    random_state: int,
) -> np.ndarray:
    '''YEAR × target 비율을 유지해 행 위치를 표본추출한다.

    Args:
        frame: YEAR와 target을 가진 DataFrame.
        sample_size: 최대 표본 수.
        random_state: 난수 시드.

    Returns:
        선택된 정수 위치 배열.
    '''
    if len(frame) <= sample_size:
        return np.arange(len(frame))
    sampled = (
        frame.assign(_position=np.arange(len(frame)))
        .groupby(["YEAR", "target"], group_keys=False)
        .apply(
            lambda group: group.sample(
                n=max(
                    1,
                    round(sample_size * len(group) / len(frame)),
                ),
                random_state=random_state,
            )
        )
    )
    positions = sampled["_position"].to_numpy()
    if len(positions) > sample_size:
        positions = np.random.default_rng(random_state).choice(
            positions,
            size=sample_size,
            replace=False,
        )
    return np.sort(positions)


shap_reference = split_assignment.loc[
    validation_mask,
    ["YEAR", "ID", "target"],
].reset_index(drop=True)
shap_positions = stratified_sample_positions(
    shap_reference,
    SHAP_SAMPLE_SIZE,
    RANDOM_STATE,
)
atomic_csv(
    shap_reference.iloc[shap_positions],
    TABLES_DIR / "shap_validation_sample_keys.csv",
)

/tmp/ipykernel_1189/2615916980.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [45]:
def dense_array(values: object) -> np.ndarray:
    '''희소행렬 또는 배열을 밀집 ndarray로 변환한다.

    Args:
        values: 변환할 행렬.

    Returns:
        밀집 ndarray.
    '''
    if sparse.issparse(values):
        return values.toarray()
    return np.asarray(values)


def normalize_shap_values(values: object) -> np.ndarray:
    '''SHAP 출력 형식을 양성 클래스 2차원 배열로 통일한다.

    Args:
        values: explainer가 반환한 SHAP 값.

    Returns:
        표본 × 피처의 양성 클래스 SHAP 배열.
    '''
    if isinstance(values, list):
        values = values[-1]
    array = np.asarray(values)
    if array.ndim == 3:
        array = array[:, :, -1]
    if array.ndim != 2:
        raise ValueError(f"예상하지 못한 SHAP 배열 형태: {array.shape}")
    return array


def original_feature_name(
    transformed_name: str,
    original_features: list[str],
) -> str:
    '''변환 피처 이름을 가장 긴 일치 원피처로 되돌린다.

    Args:
        transformed_name: ColumnTransformer 출력 피처명.
        original_features: 원피처 목록.

    Returns:
        대응하는 원피처 이름.
    '''
    stripped = transformed_name.split("__", 1)[-1]
    matches = [
        feature
        for feature in original_features
        if stripped == feature or stripped.startswith(f"{feature}_")
    ]
    if not matches:
        return stripped
    return max(matches, key=len)


def explain_pipeline(
    tier: str,
    model_name: str,
    pipeline: Pipeline,
    X_train_values: pd.DataFrame,
    X_validation_values: pd.DataFrame,
    feature_names: list[str],
    sample_positions: np.ndarray,
) -> dict:
    '''적합 Pipeline의 Validation SHAP과 집계표를 저장한다.

    Args:
        tier: Base 또는 Full.
        model_name: 모델 식별자.
        pipeline: Train 적합 Pipeline.
        X_train_values: Train 원피처.
        X_validation_values: Validation 원피처.
        feature_names: 원피처 목록.
        sample_positions: 공통 Validation 표본 위치.

    Returns:
        SHAP 배열과 집계표.
    '''
    transformed_train = pipeline[:-1].transform(X_train_values)
    transformed_validation = pipeline[:-1].transform(
        X_validation_values.iloc[sample_positions]
    )
    transformed_feature_names = pipeline.named_steps[
        "preprocess"
    ].get_feature_names_out()
    model = pipeline.named_steps["model"]
    if model_name == "elastic_net":
        background_positions = np.linspace(
            0,
            transformed_train.shape[0] - 1,
            min(500, transformed_train.shape[0]),
            dtype=int,
        )
        explainer = shap.LinearExplainer(
            model,
            dense_array(transformed_train[background_positions]),
        )
        shap_values = normalize_shap_values(
            explainer.shap_values(
                dense_array(transformed_validation)
            )
        )
        explained_values = dense_array(transformed_validation)
    else:
        explainer = shap.TreeExplainer(model)
        explained_values = dense_array(transformed_validation)
        shap_values = normalize_shap_values(
            explainer.shap_values(explained_values)
        )

    original_names = [
        original_feature_name(name, feature_names)
        for name in transformed_feature_names
    ]
    feature_summary = pd.DataFrame(
        {
            "transformed_feature": transformed_feature_names,
            "original_feature": original_names,
            "mean_abs_shap": np.abs(shap_values).mean(axis=0),
            "mean_shap": shap_values.mean(axis=0),
        }
    )
    original_summary = (
        feature_summary.groupby("original_feature", as_index=False)
        .agg(
            mean_abs_shap=("mean_abs_shap", "sum"),
            mean_shap=("mean_shap", "sum"),
        )
        .sort_values("mean_abs_shap", ascending=False)
    )
    domain_map = feature_dictionary.set_index("feature")[
        "domain"
    ].to_dict()
    original_summary["domain"] = original_summary[
        "original_feature"
    ].map(domain_map).fillna("기타")
    original_summary["direction"] = np.where(
        original_summary["mean_shap"].ge(0),
        "CASE 2",
        "CASE 1",
    )
    domain_summary = (
        original_summary.groupby("domain", as_index=False)
        .agg(mean_abs_shap=("mean_abs_shap", "sum"))
        .sort_values("mean_abs_shap", ascending=False)
    )

    prefix = f"{tier.lower()}_{model_name}"
    atomic_npz(
        MODELS_DIR / f"{prefix}_shap_values.npz",
        shap_values=shap_values,
        explained_values=explained_values,
        transformed_feature_names=np.asarray(
            transformed_feature_names,
            dtype=str,
        ),
        sample_positions=sample_positions,
    )
    atomic_csv(
        feature_summary,
        TABLES_DIR / f"{prefix}_shap_transformed_features.csv",
    )
    atomic_csv(
        original_summary,
        TABLES_DIR / f"{prefix}_shap_original_features.csv",
    )
    atomic_csv(
        domain_summary,
        TABLES_DIR / f"{prefix}_shap_domains.csv",
    )
    return {
        "values": shap_values,
        "data": explained_values,
        "feature_names": transformed_feature_names,
        "original_summary": original_summary,
        "domain_summary": domain_summary,
    }

In [46]:
shap_results = {}
for tier, best_row in [
    ("Base", best_base_row),
    ("Full", best_full_row),
]:
    model_name = best_row["model"]
    features = base_features if tier == "Base" else full_features
    data = base_data if tier == "Base" else full_data
    pipeline = joblib.load(
        MODELS_DIR
        / f"{tier.lower()}_{model_name}_train_pipeline.joblib"
    )
    shap_results[tier] = explain_pipeline(
        tier,
        model_name,
        pipeline,
        data.loc[train_mask, features],
        data.loc[validation_mask, features],
        features,
        shap_positions,
    )
    print(f"{tier} SHAP 저장 완료")

Base SHAP 저장 완료


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Full SHAP 저장 완료


## 15. 관계 직접신호 제거 민감도

최종 선택 알고리즘과 같은 하이퍼파라미터만 사용한다. 가족·친지집 숙박
코드 12 및 이를 포함한 파생변수, `A6B_1`, `A6B_2`, `A6B_3`을 제거하고
Train에 다시 적합한다. Validation 임계값은 다시 고르되, 모델 경쟁과
Test 평가에는 포함하지 않는다.

In [47]:
direct_signal_features = {
    "family_home_only",
    "has_lodging_12",
    "A6B_1",
    "A6B_2",
    "A6B_3",
}
sensitivity_features = [
    feature
    for feature in selected_features
    if feature not in direct_signal_features
]
sensitivity_pipeline = build_pipeline(
    clone(specifications[selected_model_name]["estimator"]),
    sensitivity_features,
)
sensitivity_pipeline.set_params(
    **base_best_parameters[selected_model_name]
)
sensitivity_pipeline.fit(
    selected_data.loc[train_mask, sensitivity_features],
    y_train,
)
sensitivity_probabilities = sensitivity_pipeline.predict_proba(
    selected_data.loc[validation_mask, sensitivity_features]
)[:, 1]
sensitivity_threshold, sensitivity_threshold_table = select_threshold(
    y_validation,
    sensitivity_probabilities,
)
sensitivity_metrics = binary_metrics(
    y_validation,
    sensitivity_probabilities,
    sensitivity_threshold,
)
baseline_metrics = {
    key: float(selected_row[key])
    for key in ["f1", "precision", "recall", "pr_auc", "roc_auc"]
}
sensitivity_comparison = pd.DataFrame(
    [
        {
            "configuration": "본모형",
            "threshold": selected_threshold,
            **baseline_metrics,
        },
        {
            "configuration": "직접신호 제거",
            "threshold": sensitivity_threshold,
            **sensitivity_metrics,
        },
    ]
)
for metric in ["f1", "pr_auc"]:
    sensitivity_comparison[f"{metric}_change_from_baseline"] = (
        sensitivity_comparison[metric]
        - sensitivity_comparison.loc[
            sensitivity_comparison["configuration"].eq("본모형"),
            metric,
        ].iloc[0]
    )

atomic_joblib(
    sensitivity_pipeline,
    MODELS_DIR / "sensitivity_pipeline.joblib",
)
atomic_csv(
    sensitivity_threshold_table,
    TABLES_DIR / "sensitivity_validation_thresholds.csv",
)
atomic_csv(
    sensitivity_comparison,
    TABLES_DIR / "sensitivity_comparison.csv",
)
display(sensitivity_comparison)

,configuration,threshold,f1,precision,recall,pr_auc,roc_auc,f1_change_from_baseline,pr_auc_change_from_baseline
0,본모형,0.615,0.687812,0.788022,0.610214,0.729890,0.873743,0.000000,0.000000
1,직접신호 제거,0.600,0.689855,0.777270,0.620115,0.729757,0.873671,0.002043,-0.000133


In [48]:
sensitivity_shap = explain_pipeline(
    "Sensitivity",
    selected_model_name,
    sensitivity_pipeline,
    selected_data.loc[train_mask, sensitivity_features],
    selected_data.loc[validation_mask, sensitivity_features],
    sensitivity_features,
    shap_positions,
)
baseline_top20 = set(
    shap_results[selected_tier]["original_summary"]
    .head(20)["original_feature"]
)
sensitivity_top20 = set(
    sensitivity_shap["original_summary"]
    .head(20)["original_feature"]
)
domain_rank_baseline = {
    domain: rank
    for rank, domain in enumerate(
        shap_results[selected_tier]["domain_summary"]["domain"],
        start=1,
    )
}
domain_rank_sensitivity = {
    domain: rank
    for rank, domain in enumerate(
        sensitivity_shap["domain_summary"]["domain"],
        start=1,
    )
}
domain_rank_change = pd.DataFrame(
    {
        "domain": sorted(
            set(domain_rank_baseline) | set(domain_rank_sensitivity)
        )
    }
)
domain_rank_change["baseline_rank"] = domain_rank_change[
    "domain"
].map(domain_rank_baseline)
domain_rank_change["sensitivity_rank"] = domain_rank_change[
    "domain"
].map(domain_rank_sensitivity)
domain_rank_change["rank_change"] = (
    domain_rank_change["sensitivity_rank"]
    - domain_rank_change["baseline_rank"]
)
sensitivity_shap_summary = pd.DataFrame(
    [
        {
            "top20_overlap_N": len(
                baseline_top20 & sensitivity_top20
            ),
            "top20_jaccard": len(
                baseline_top20 & sensitivity_top20
            )
            / len(baseline_top20 | sensitivity_top20),
        }
    ]
)
atomic_csv(
    sensitivity_shap_summary,
    TABLES_DIR / "sensitivity_shap_top20_overlap.csv",
)
atomic_csv(
    domain_rank_change,
    TABLES_DIR / "sensitivity_domain_rank_change.csv",
)

/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


## 16. 표와 그림

모든 그림은 프로젝트 색상과 Pretendard를 적용하고 900 DPI로 저장한다.
제목·부제에는 2023~2025년, 비가중 분석과 분모 N을 표시한다. 색상 외에
정렬, 직접 레이블, 선 모양을 함께 사용한다.

In [49]:
def save_figure(
    figure: plt.Figure,
    filename: str,
) -> None:
    '''그림을 900 DPI PNG로 저장하고 현재 Figure를 닫는다.

    Args:
        figure: 저장할 Matplotlib Figure.
        filename: 영문 snake_case PNG 파일명.
    '''
    figure.savefig(
        FIGURES_DIR / filename,
        dpi=900,
        bbox_inches="tight",
    )
    plt.close(figure)


figure, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4.8),
    gridspec_kw={"width_ratios": [1, 1.4]},
)
axes[0].barh(
    sample_flow["단계"],
    sample_flow["비가중_N"],
    color=[
        COLORS["neutral"],
        COLORS["primary"],
        COLORS["neutral"],
        COLORS["secondary"],
    ],
)
for index, value in enumerate(sample_flow["비가중_N"]):
    axes[0].text(value, index, f" {value:,}", va="center")
axes[0].set_title("표본 흐름")
axes[0].set_xlabel("비가중 N")

year_pivot = year_case.pivot(
    index="YEAR",
    columns="CASE",
    values="연도내_비율",
).fillna(0)
axes[1].bar(
    year_pivot.index.astype(str),
    year_pivot["CASE 1"],
    label="CASE 1",
    color=COLORS["primary"],
)
axes[1].bar(
    year_pivot.index.astype(str),
    year_pivot["CASE 2"],
    bottom=year_pivot["CASE 1"],
    label="CASE 2",
    color=COLORS["secondary"],
)
axes[1].set_title("연도별 CASE 구성")
axes[1].set_ylabel("구성비")
axes[1].legend(frameon=False)
figure.suptitle(
    f"2023~2025년 CASE 1·2 표본 구성 | 비가중 N={len(analysis_data):,}"
)
figure.tight_layout()
save_figure(figure, "sample_flow_and_year_case_composition.png")

/tmp/ipykernel_1189/3213786091.py:64: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/3213786091.py:64: UserWarning: Glyph 44032 (\N{HANGUL SYLLABLE GA}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/3213786091.py:64: UserWarning: Glyph 51473 (\N{HANGUL SYLLABLE JUNG}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/3213786091.py:64: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/3213786091.py:64: UserWarning: Glyph 52404 (\N{HANGUL SYLLABLE CE}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/3213786091.py:64: UserWarning: Glyph 51025 (\N{HANGUL SYLLABLE EUNG}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/3213786091.py:64: UserWarning: Glyph 45813 (\N{HANGUL SYLLABLE DAB}) missing from font(s) DejaVu S

In [50]:
leaderboard_plot = validation_leaderboard.copy()
leaderboard_plot["label"] = (
    leaderboard_plot["tier"]
    + " · "
    + leaderboard_plot["model"]
)
leaderboard_plot = leaderboard_plot.sort_values("f1")
colors = leaderboard_plot["tier"].map(
    {"Base": COLORS["primary"], "Full": COLORS["secondary"]}
)
figure, axis = plt.subplots(figsize=(9, 6))
axis.barh(
    leaderboard_plot["label"],
    leaderboard_plot["f1"],
    color=colors,
)
for index, value in enumerate(leaderboard_plot["f1"]):
    axis.text(value, index, f" {value:.3f}", va="center")
axis.set_xlim(0, max(1.0, leaderboard_plot["f1"].max() + 0.08))
axis.set_xlabel("Validation F1")
axis.set_title(
    f"2023~2025년 Base/Full Validation 성능 | 비가중 N={validation_mask.sum():,}"
)
figure.tight_layout()
save_figure(figure, "validation_leaderboard_f1.png")

figure, axis = plt.subplots(figsize=(8, 4.8))
axis.hist(
    bootstrap_results["full_minus_base_f1"],
    bins=40,
    color=COLORS["secondary"],
    alpha=0.85,
)
axis.axvline(0, color=COLORS["neutral"], linestyle="--")
axis.axvline(
    bootstrap_summary["ci_2_5"].iloc[0],
    color=COLORS["accent"],
    linestyle=":",
)
axis.axvline(
    bootstrap_summary["ci_97_5"].iloc[0],
    color=COLORS["accent"],
    linestyle=":",
)
axis.set_xlabel("Full − Base F1")
axis.set_ylabel("Bootstrap 빈도")
axis.set_title(
    "2023~2025년 paired bootstrap | "
    f"Validation 비가중 N={validation_mask.sum():,}, "
    f"{BOOTSTRAP_ITERATIONS:,}회"
)
figure.tight_layout()
save_figure(figure, "paired_bootstrap_f1_difference.png")

/tmp/ipykernel_1189/4033940248.py:24: UserWarning: Glyph 45380 (\N{HANGUL SYLLABLE NYEON}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/4033940248.py:24: UserWarning: Glyph 49457 (\N{HANGUL SYLLABLE SEONG}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/4033940248.py:24: UserWarning: Glyph 45733 (\N{HANGUL SYLLABLE NEUNG}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/4033940248.py:24: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/4033940248.py:24: UserWarning: Glyph 44032 (\N{HANGUL SYLLABLE GA}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/4033940248.py:24: UserWarning: Glyph 51473 (\N{HANGUL SYLLABLE JUNG}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/3213786091.py:11: UserWarning: Glyph 45380 (\N{HANGUL SYLLABLE NYEON}) missing from font(s) D

In [51]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.8))
sns.heatmap(
    test_confusion_matrix,
    annot=True,
    fmt=",d",
    cmap=sns.light_palette(COLORS["primary"], as_cmap=True),
    cbar=False,
    xticklabels=["CASE 1", "CASE 2"],
    yticklabels=["CASE 1", "CASE 2"],
    ax=axes[0],
)
axes[0].set_xlabel("예측")
axes[0].set_ylabel("실제")
axes[0].set_title("최종 Test 혼동행렬")

precision_values, recall_values, _ = precision_recall_curve(
    y_test,
    test_probabilities,
)
axes[1].plot(
    recall_values,
    precision_values,
    color=COLORS["primary"],
    linewidth=2,
    label=f"PR-AUC={test_metrics['pr_auc']:.3f}",
)
axes[1].axhline(
    y_test.mean(),
    color=COLORS["neutral"],
    linestyle="--",
    label=f"양성 비율={y_test.mean():.3f}",
)
axes[1].set_xlabel("재현율")
axes[1].set_ylabel("정밀도")
axes[1].set_title("최종 Test PR 곡선")
axes[1].legend(frameon=False)
figure.suptitle(
    f"2023~2025년 최종 Test | 비가중 N={len(y_test):,}"
)
figure.tight_layout()
save_figure(figure, "final_test_confusion_matrix_and_pr_curve.png")

/tmp/ipykernel_1189/2815211038.py:40: UserWarning: Glyph 50696 (\N{HANGUL SYLLABLE YE}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/2815211038.py:40: UserWarning: Glyph 52769 (\N{HANGUL SYLLABLE CEUG}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/2815211038.py:40: UserWarning: Glyph 49892 (\N{HANGUL SYLLABLE SIL}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/2815211038.py:40: UserWarning: Glyph 51228 (\N{HANGUL SYLLABLE JE}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/2815211038.py:40: UserWarning: Glyph 52572 (\N{HANGUL SYLLABLE COE}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/2815211038.py:40: UserWarning: Glyph 51333 (\N{HANGUL SYLLABLE JONG}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/2815211038.py:40: UserWarning: Glyph 54844 (\N{HANGUL SYLLABLE HON}) missing from font(s) DejaVu S

In [52]:
for tier, result in shap_results.items():
    top_original = result["original_summary"].head(20).sort_values(
        "mean_abs_shap"
    )
    figure, axis = plt.subplots(figsize=(8, 7))
    axis.barh(
        top_original["original_feature"],
        top_original["mean_abs_shap"],
        color=(
            COLORS["primary"]
            if tier == "Base"
            else COLORS["secondary"]
        ),
    )
    axis.set_xlabel("mean |SHAP|")
    axis.set_title(
        f"2023~2025년 {tier} 상위 20개 피처 | "
        f"Validation 비가중 N={len(shap_positions):,}"
    )
    figure.tight_layout()
    save_figure(
        figure,
        f"{tier.lower()}_shap_original_feature_bar.png",
    )

    shap.summary_plot(
        result["values"],
        result["data"],
        feature_names=result["feature_names"],
        max_display=20,
        show=False,
    )
    figure = plt.gcf()
    figure.suptitle(
        f"2023~2025년 {tier} SHAP beeswarm | "
        f"Validation 비가중 N={len(shap_positions):,}",
        y=1.02,
    )
    save_figure(
        figure,
        f"{tier.lower()}_shap_beeswarm.png",
    )

full_domain = shap_results["Full"]["domain_summary"].sort_values(
    "mean_abs_shap"
)
figure, axis = plt.subplots(figsize=(8, 5))
axis.barh(
    full_domain["domain"],
    full_domain["mean_abs_shap"],
    color=COLORS["secondary"],
)
axis.set_xlabel("도메인 합산 mean |SHAP|")
axis.set_title(
    "2023~2025년 Full 도메인 중요도 | "
    f"Validation 비가중 N={len(shap_positions):,}"
)
figure.tight_layout()
save_figure(figure, "full_shap_domain_importance.png")

/tmp/ipykernel_1189/4076831533.py:20: UserWarning: Glyph 45380 (\N{HANGUL SYLLABLE NYEON}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/4076831533.py:20: UserWarning: Glyph 49345 (\N{HANGUL SYLLABLE SANG}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/4076831533.py:20: UserWarning: Glyph 50948 (\N{HANGUL SYLLABLE WI}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/4076831533.py:20: UserWarning: Glyph 44060 (\N{HANGUL SYLLABLE GAE}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/4076831533.py:20: UserWarning: Glyph 54588 (\N{HANGUL SYLLABLE PI}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/4076831533.py:20: UserWarning: Glyph 52376 (\N{HANGUL SYLLABLE CEO}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/4076831533.py:20: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu S

In [53]:
sensitivity_plot = sensitivity_comparison.melt(
    id_vars="configuration",
    value_vars=["f1", "pr_auc"],
    var_name="metric",
    value_name="value",
)
figure, axis = plt.subplots(figsize=(7.5, 4.8))
sns.barplot(
    data=sensitivity_plot,
    x="metric",
    y="value",
    hue="configuration",
    palette=[COLORS["primary"], COLORS["accent"]],
    ax=axis,
)
axis.set_ylim(0, 1)
axis.set_xlabel("")
axis.set_ylabel("Validation 지표")
axis.set_title(
    "2023~2025년 직접 연고신호 제거 민감도 | "
    f"비가중 N={validation_mask.sum():,}"
)
axis.legend(frameon=False)
figure.tight_layout()
save_figure(figure, "direct_signal_sensitivity_comparison.png")

/tmp/ipykernel_1189/1476493362.py:24: UserWarning: Glyph 51648 (\N{HANGUL SYLLABLE JI}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/1476493362.py:24: UserWarning: Glyph 54364 (\N{HANGUL SYLLABLE PYO}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/1476493362.py:24: UserWarning: Glyph 45380 (\N{HANGUL SYLLABLE NYEON}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/1476493362.py:24: UserWarning: Glyph 51649 (\N{HANGUL SYLLABLE JIG}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/1476493362.py:24: UserWarning: Glyph 51217 (\N{HANGUL SYLLABLE JEOB}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/1476493362.py:24: UserWarning: Glyph 50672 (\N{HANGUL SYLLABLE YEON}) missing from font(s) DejaVu Sans.
  figure.tight_layout()
/tmp/ipykernel_1189/1476493362.py:24: UserWarning: Glyph 44256 (\N{HANGUL SYLLABLE GO}) missing from font(s) DejaVu

## 17. 완료 검증

저장한 전처리 파일·분할·모델·SHAP을 다시 열고 다음을 확인한다.

- Base/Full의 키·타깃·split 완전 일치
- `WT_DOM`의 모델 입력 미사용
- 확률·평가지표의 유효 범위와 혼동행렬 합계
- Pipeline의 Train 전용 지출 상한 존재
- Parquet·joblib·NPZ 재로딩 가능

In [54]:
reloaded_base = pd.read_parquet(BASE_PARQUET_PATH)
reloaded_full = pd.read_parquet(FULL_PARQUET_PATH)
reloaded_split = pd.read_csv(SPLIT_ASSIGNMENT_PATH)
reloaded_final_pipeline = joblib.load(
    MODELS_DIR / "final_pipeline.joblib"
)
reloaded_shap = np.load(
    MODELS_DIR
    / (
        f"{best_full_row['tier'].lower()}_"
        f"{best_full_row['model']}_shap_values.npz"
    ),
    allow_pickle=False,
)

assert reloaded_base[key_columns].equals(
    reloaded_full[key_columns]
)
assert reloaded_base["target"].equals(reloaded_full["target"])
assert reloaded_split[key_columns].equals(
    reloaded_base[key_columns]
)
assert "WT_DOM" not in selected_features
assert np.isfinite(test_probabilities).all()
assert np.logical_and(
    test_probabilities >= 0,
    test_probabilities <= 1,
).all()
assert all(
    0 <= value <= 1 for value in test_metrics.values()
)
assert test_confusion_matrix.sum() == len(y_test)
assert (
    reloaded_final_pipeline.named_steps["spend"].upper_bound_
    is not None
)
assert reloaded_shap["shap_values"].shape[0] == len(
    shap_positions
)

run_metadata["completed_at"] = datetime.now().isoformat(
    timespec="seconds"
)
run_metadata["status"] = "completed"
run_metadata["selected_configuration"] = {
    "tier": selected_tier,
    "model": selected_model_name,
    "threshold": selected_threshold,
}
atomic_json(run_metadata, MODELS_DIR / "run_metadata.json")
print("모든 완료 검증을 통과했습니다.")

모든 완료 검증을 통과했습니다.


## 18. 핵심 결과 기록과 한계

실행 후 `validation_leaderboard.csv`, `paired_bootstrap_summary.csv`,
`final_test_metrics.csv`, Base/Full SHAP 및 민감도 표를 함께 읽어
결과를 기록한다. 실행 전에는 계산되지 않은 수치를 미리 서술하지 않는다.

해석 시 반드시 다음 한계를 함께 제시한다.

- Full은 Base 최적 하이퍼파라미터를 재사용해 계산예산을 줄였으므로
  Base보다 불리할 수 있다.
- 표본 기반 비가중 분류 성능이며 모집단 효과 추정이 아니다.
- SHAP은 모델 안의 연관성과 방향을 요약하며 인과효과가 아니다.
- 직접신호 제거 민감도는 Validation 진단이며 모델 경쟁이나 Test 성능으로
  사용하지 않는다.

In [55]:
if not COLAB_MODE:
    from src.notebook_sync import sync_script_from_notebook

    sync_script_from_notebook(
        "notebooks/03_Classification/"
        "260726_CASE1_CASE2_분류모형.ipynb"
    )
else:
    print("Colab에서는 로컬 생성 스크립트 동기화를 건너뜁니다.")

Colab에서는 로컬 생성 스크립트 동기화를 건너뜁니다.
